# EEGSynthesizer — validación externa reproducible

Este notebook separa estrictamente desarrollo y evaluación externa. Los casos
`chb01`, `chb02`, `chb03`, `chb05`, `chb06` y el registro `chb21` —la misma
persona que `chb01`— se reservan para desarrollo. Los demás sujetos permanecen
bloqueados hasta que el generador tenga un manifiesto congelado.

## Cómo ejecutar

Cambie únicamente `NOTEBOOK_MODE` en la primera celda de código y use **Run All**:

1. `status`: inspecciona prerrequisitos sin descargar ni recalcular.
2. `prepare_dev`: descarga/reutiliza CHB-MIT de desarrollo y crea el perfil 5.1.
3. Después ejecute `1_0_EEGSynthesizer_DATASET.ipynb` en modo `full`.
4. `external`: construye/reutiliza CHB-MIT y ejecuta la evaluación externa bloqueada.
5. `siena_external`: descarga/reutiliza Siena y ejecuta la validación multicorpus.

Las descargas usan archivos `.part`, reanudación HTTP, reintentos y SHA-256. Si un
corpus ya está completo, el notebook lo informa y no vuelve a descargarlo. Los
resultados derivados se reutilizan únicamente cuando sus manifiestos y hashes son
válidos. Una reconstrucción completa requiere aproximadamente 60 GiB libres.

Los nombres `generalized_absence` y `focal_temporal` son escenarios paramétricos,
no diagnósticos. CHB-MIT y Siena permiten evaluar plausibilidad morfológica
operacional, cobertura por rasgos y utilidad de transferencia en dominios externos.
Una afirmación de equivalencia clínica requeriría otro diseño, con márgenes clínicos
predefinidos y evaluación experta independiente. La proximidad se interpreta
exclusivamente como una medida descriptiva en el espacio de características.

## Interpretación de estados técnicos

- `SUPPORTED`: se cumplió el criterio prospectivo especificado.
- `NOT SUPPORTED`: no se alcanzó el criterio completo; no significa ausencia de
  efecto, daño ni invalidez del sintetizador.
- `NOT EVALUABLE`: el tamaño muestral quedó por debajo del mínimo prospectivo; la
  estimación se reporta como exploratoria.
- `REPORTED`: referencia descriptiva sin decisión confirmatoria asociada.


In [ ]:
# BLOQUE 0 — configuración visible, rutas y preflight
import concurrent.futures, gc, hashlib, importlib.metadata, json, os, re, shutil, subprocess, sys, time, urllib.error, urllib.request
from pathlib import Path

# Reducciones y modelos en un solo hilo para reproducibilidad numérica entre
# ejecuciones del mismo entorno. Las semillas de cada sujeto se fijan después.
os.environ.setdefault("OMP_NUM_THREADS","1")
os.environ.setdefault("MKL_NUM_THREADS","1")
os.environ.setdefault("OPENBLAS_NUM_THREADS","1")
os.environ.setdefault("NUMEXPR_NUM_THREADS","1")

import matplotlib
matplotlib.use("Agg")
import numpy as np
import pandas as pd
import mne
from scipy import signal, stats
from scipy.spatial.distance import cdist
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.neighbors import NearestNeighbors
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Selección interactiva. La variable del sistema es solo una alternativa para
# ejecución automatizada; un investigador puede cambiar esta línea en Jupyter.
NOTEBOOK_MODE = "siena_external"  # status | prepare_dev | external | siena_external
MODE = os.environ.get("EEGSYN_VALIDATION_MODE", NOTEBOOK_MODE).strip().lower()
if MODE not in {"status", "prepare_dev", "external", "siena_external"}:
    raise ValueError("MODE debe ser status, prepare_dev, external o siena_external")

PROJECT_ROOT = Path.cwd().resolve()
EXPECTED_PROJECT_FILES = {"1_0_EEGSynthesizer_DATASET.ipynb", "1_1_EEGSynthesizer_VALIDATION.ipynb", "requirements.txt"}
missing_project_files = sorted(name for name in EXPECTED_PROJECT_FILES if not (PROJECT_ROOT/name).exists())
if missing_project_files:
    raise RuntimeError("Abra Jupyter desde la raíz del repositorio; faltan: " + ", ".join(missing_project_files))

SYN_DIR = PROJECT_ROOT / "dataset_eeg_final"
REAL_DIR = PROJECT_ROOT / "dataset_doctorado_final"
OUT_DIR = REAL_DIR / "validation_q1_assets"
CACHE = REAL_DIR / "chb_mit_cache"
WORK_ROOT = PROJECT_ROOT / "tmp" / "eegsynth_rebuild"
DEV_WORK = WORK_ROOT / "dev_candidate"
REAL_WORK = WORK_ROOT / "real_candidate"
STATE_PATH = REAL_DIR / "reproducibility_state.json"
OUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE.mkdir(parents=True, exist_ok=True)

REBUILD_REAL = os.environ.get("EEGSYN_REBUILD_REAL", "0") == "1"
DOWNLOAD_WORKERS = max(1, int(os.environ.get("EEGSYN_DOWNLOAD_WORKERS", "4")))
FS = 250
WIN_SEC = 2.0
WIN_PTS = int(FS * WIN_SEC)
SEED = 42
MAX_WINDOWS_PER_SUBJECT_CLASS = 400
DEV_CASES = {"chb01", "chb02", "chb03", "chb05", "chb06"}
CASE_PERSON = {"chb21": "chb01"}

REF_CH = ["Fp1","Fp2","F7","F3","Fz","F4","F8","T3","C3","Cz","C4","T4","T5","P3","Pz","P4","T6","O1","O2"]
BIPOLAR = [
    ("FP1-F7","FP1","F7"),("F7-T3","F7","T3"),("T3-T5","T3","T5"),("T5-O1","T5","O1"),
    ("FP1-F3","FP1","F3"),("F3-C3","F3","C3"),("C3-P3","C3","P3"),("P3-O1","P3","O1"),
    ("FP2-F4","FP2","F4"),("F4-C4","F4","C4"),("C4-P4","C4","P4"),("P4-O2","P4","O2"),
    ("FP2-F8","FP2","F8"),("F8-T4","F8","T4"),("T4-T6","T4","T6"),("T6-O2","T6","O2"),
    ("FZ-CZ","FZ","CZ"),("CZ-PZ","CZ","PZ"),
]
FEATURE_NAMES = ["ptp_med","ptp_p95","std_med","std_p95","bp1_4","bp4_8","bp8_13","bp13_30",
                 "ratio_2_6__6_20","beta","Hspec","corr_abs_mean","corr_abs_p95","dom_freq",
                 "freq_first","freq_last","rms_last_first","spatial_concentration","laterality_abs"]
FEATURE_DOMAINS = {
    "background":[0,1,4,5,6,7,9,10], "temporal":[8,13,14,15,16],
    "spatial":[11,12,17,18], "variability":list(range(len(FEATURE_NAMES))),
}

def atomic_write_json(path, payload):
    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True)
    part=Path(str(path)+".part")
    part.write_text(json.dumps(payload,indent=2,ensure_ascii=False)+"\n",encoding="utf-8")
    os.replace(part,path)

def record_stage(stage, status, artifacts=None, details=None):
    state={"schema_version":"1.0-reproducibility","stages":{}}
    if STATE_PATH.exists():
        try: state=json.loads(STATE_PATH.read_text(encoding="utf-8"))
        except Exception: pass
    artifact_rows={}
    for label,path in (artifacts or {}).items():
        p=Path(path)
        artifact_rows[label]={"exists":p.exists(),"bytes":p.stat().st_size if p.exists() else None,
                              "sha256":sha256_file(p) if p.exists() else None}
    state.setdefault("stages",{})[stage]={"status":status,"completed_utc":time.strftime("%Y-%m-%dT%H:%M:%SZ",time.gmtime()),
                                          "artifacts":artifact_rows,"details":details or {}}
    atomic_write_json(STATE_PATH,state)

def stage_presence():
    def complete(base,names): return all((base/name).exists() for name in names)
    return {
      "S1_development_profile": complete(OUT_DIR,["development_profile.json","dev_window_features.csv","dev_window_metadata.csv"]),
      "S3_synthetic_frozen": complete(SYN_DIR,["generation_manifest.json","X_train.npy","X_val.npy","X_test.npy",
                                                     "y_train.npy","y_val.npy","y_test.npy","cohort_metadata.csv",
                                                     "window_metadata_train.csv","window_metadata_val.csv","window_metadata_test.csv"]),
      "S4_chbmit_derived": complete(REAL_DIR,["real_manifest.json","X_real.npy","y_real.npy","pid_real.npy","real_window_metadata.csv"]),
      "S5_siena_raw_index": (REAL_DIR/"siena_scalp_cache"/"SHA256SUMS.txt").exists(),
      "S6_siena_derived": complete(REAL_DIR,["siena_manifest.json","siena_X.npy","siena_y.npy","siena_pid.npy",
                                                   "siena_window_metadata.csv","siena_annotation_audit.csv"]),
      "S7_external_result_snapshots": complete(OUT_DIR,["05_final_summary.json","13_multicorpus_final_summary.json",
                                                       "14_siena_ictal_preictal_integrity.json"]),
    }

versions={name:importlib.metadata.version(name) for name in
          ("numpy","pandas","scipy","mne","scikit-learn")}
free_gib=shutil.disk_usage(PROJECT_ROOT).free/(1024**3)
print("MODE:",MODE,"REBUILD_REAL:",REBUILD_REAL,"DOWNLOAD_WORKERS:",DOWNLOAD_WORKERS)
print("Entorno:",sys.version.split()[0],versions)
print(f"Espacio libre: {free_gib:.2f} GiB")
print("Estado de etapas:",json.dumps(stage_presence(),indent=2,ensure_ascii=False))


In [ ]:
# BLOQUE 1 — integridad, montaje común y características
def sha256_file(path, block=16*1024*1024):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for b in iter(lambda:f.read(block),b""): h.update(b)
    return h.hexdigest()

def stable_int(value):
    return int.from_bytes(hashlib.sha256(str(value).encode()).digest()[:4],"little")

def load_synthetic(require_frozen=False):
    mp=SYN_DIR/"generation_manifest.json"
    required=["X_train.npy","X_val.npy","X_test.npy","y_train.npy","y_val.npy","y_test.npy",
              "cohort_metadata.csv","window_metadata_train.csv","window_metadata_val.csv","window_metadata_test.csv"]
    if not mp.exists(): return {"ready":False,"reason":"falta generation_manifest.json"}
    manifest=json.loads(mp.read_text(encoding="utf-8"))
    if int(str(manifest.get("config",{}).get("schema_version","0")).split(".")[0])<4:
        return {"ready":False,"reason":"schema sintético anterior a 4"}
    missing=[n for n in required if not (SYN_DIR/n).exists()]
    if missing: return {"ready":False,"reason":f"faltan artefactos: {missing}"}
    bad=[]
    for rel,info in manifest.get("files",{}).items():
        p=SYN_DIR/rel
        if not p.exists() or sha256_file(p)!=info.get("sha256"): bad.append(rel)
    if bad: return {"ready":False,"reason":f"SHA256 inválido: {bad[:8]}"}
    if require_frozen and not manifest.get("external_freeze",{}).get("frozen",False):
        return {"ready":False,"reason":"el generador no fue congelado antes de evaluación externa"}
    return {"ready":True,"manifest":manifest,
            "X":{k:np.load(SYN_DIR/f"X_{k}.npy",mmap_mode="r") for k in ("train","val","test")},
            "y":{k:np.load(SYN_DIR/f"y_{k}.npy") for k in ("train","val","test")},
            "meta":{k:pd.read_csv(SYN_DIR/f"window_metadata_{k}.csv") for k in ("train","val","test")},
            "cohort":pd.read_csv(SYN_DIR/"cohort_metadata.csv")}

def synthetic_to_bipolar(X):
    idx={c.upper():i for i,c in enumerate(REF_CH)}
    return np.stack([np.asarray(X)[:,:,idx[a]]-np.asarray(X)[:,:,idx[b]] for _,a,b in BIPOLAR],axis=2).astype(np.float32)

def norm_name(x):
    x=x.upper().replace("EEG","").replace("-REF","").replace(" ","")
    x=re.sub(r"-(?:0|1)$","",x)
    return x.replace("T7","T3").replace("P7","T5").replace("T8","T4").replace("P8","T6")

def common_bipolar(data_ct,names):
    lookup={}
    for i,name in enumerate(names):
        q=norm_name(name)
        if "-" in q:
            a,b=q.split("-",1)
            if (a,b) not in lookup:
                lookup[(a,b)]=(i,1.0); lookup[(b,a)]=(i,-1.0)
    out=[]
    for _,a,b in BIPOLAR:
        if (a,b) not in lookup: return None
        i,sgn=lookup[(a,b)]; out.append(sgn*data_ct[i])
    return np.asarray(out,dtype=np.float32).T

def one_feature(window,zscore=False):
    x=np.asarray(window,dtype=np.float64)
    if zscore: x=(x-x.mean(0,keepdims=True))/(x.std(0,keepdims=True)+1e-8)
    ptp=np.ptp(x,axis=0); sd=x.std(axis=0)
    f,pch=signal.welch(x,fs=FS,nperseg=min(256,len(x)),axis=0); p=pch.mean(axis=1)
    def bp(a,b):
        m=(f>=a)&(f<b); return float(np.trapezoid(p[m],f[m])) if m.sum()>=2 else 0.0
    m=(f>=1)&(f<=30)&(p>0); beta=float(np.polyfit(np.log(f[m]),np.log(p[m]),1)[0]) if m.sum()>=3 else 0.0
    pn=p/(p.sum()+1e-12); H=float(-(pn[pn>0]*np.log(pn[pn>0])).sum()/np.log(max(2,len(pn))))
    C=np.corrcoef(x,rowvar=False); cv=np.abs(C[np.triu_indices_from(C,k=1)]); cv=cv[np.isfinite(cv)]
    band=(f>=1)&(f<=20); dom=float(f[band][np.argmax(p[band])]) if band.any() else 0.0
    third=max(FS//2,len(x)//3)
    def dompart(q):
        fq,pq=signal.welch(q,fs=FS,nperseg=min(256,len(q)),axis=0); pq=pq.mean(1); mm=(fq>=1)&(fq<=20)
        return float(fq[mm][np.argmax(pq[mm])]) if mm.any() else 0.0
    ff,fl=dompart(x[:third]),dompart(x[-third:])
    rf=float(np.sqrt(np.mean(x[-third:]**2))/(np.sqrt(np.mean(x[:third]**2))+1e-12))
    concentration=float(ptp.max()/(ptp.sum()+1e-12))
    left=float(ptp[:8].sum()); right=float(ptp[8:16].sum()); lateral=abs(left-right)/(left+right+1e-12)
    return [float(np.median(ptp)),float(np.percentile(ptp,95)),float(np.median(sd)),float(np.percentile(sd,95)),
            bp(1,4),bp(4,8),bp(8,13),bp(13,30),bp(2,6)/(bp(6,20)+1e-12),beta,H,
            float(cv.mean()) if len(cv) else 0.0,float(np.percentile(cv,95)) if len(cv) else 0.0,
            dom,ff,fl,rf,concentration,float(lateral)]

def extract_features(X,zscore=False,tag=""):
    F=np.asarray([one_feature(w,zscore=zscore) for w in X],dtype=np.float64)
    if not np.isfinite(F).all(): raise ValueError(f"NaN/Inf en características {tag}")
    print("features",tag,F.shape); return F

SYN=load_synthetic(require_frozen=(MODE in {"external","siena_external"}))
print("SYN:","SUPPORTED" if SYN["ready"] else "NOT EVALUABLE — "+SYN["reason"])

In [ ]:
# BLOQUE 2 — selección oficial y descarga reanudable de CHB-MIT
BASE_URL="https://physionet.org/files/chbmit/1.0.0/"
DATA_URL="https://physionet-open.s3.amazonaws.com/chbmit/1.0.0/"

def fetch_text(name,retries=5):
    for attempt in range(retries):
        try: return urllib.request.urlopen(BASE_URL+name,timeout=90).read().decode("utf-8",errors="replace")
        except Exception:
            if attempt+1==retries: raise
            time.sleep(min(30,2**attempt))

def parse_summary(text):
    events={}; current=None; start=None
    for line in text.splitlines():
        mf=re.search(r"File Name:\s*(\S+)",line,re.I)
        if mf: current=mf.group(1); start=None
        ms=re.search(r"Seizure(?: \d+)? Start Time:\s*([0-9.]+)",line,re.I)
        me=re.search(r"Seizure(?: \d+)? End Time:\s*([0-9.]+)",line,re.I)
        if ms: start=float(ms.group(1))
        if me and current and start is not None:
            events.setdefault(current,[]).append((start,float(me.group(1))))
    return events

def official_index(scope):
    records=[x.strip() for x in fetch_text("RECORDS").splitlines() if x.strip().endswith(".edf")]
    seizure_records=set(x.strip() for x in fetch_text("RECORDS-WITH-SEIZURES").splitlines() if x.strip())
    checksums={}
    for line in fetch_text("SHA256SUMS.txt").splitlines():
        m=re.match(r"([0-9a-fA-F]{64})\s+\*?(.+)",line.strip())
        if m: checksums[m.group(2).lstrip("./")]=m.group(1).lower()
    by={}
    for rel in records: by.setdefault(rel.split("/")[0],[]).append(rel)
    selected=[]; events={}
    for case,rels in sorted(by.items()):
        person=CASE_PERSON.get(case,case); is_dev=person in DEV_CASES
        if scope=="dev" and not is_dev: continue
        ictal=sorted(r for r in rels if r in seizure_records)
        if not ictal: continue
        controls=sorted(r for r in rels if r not in seizure_records)[:(2 if is_dev else 1)]
        selected.extend(ictal+controls)
        summary=parse_summary(fetch_text(f"{case}/{case}-summary.txt"))
        for rel in ictal: events[rel]=summary.get(Path(rel).name,[])
    core=[r for r in records if not r.startswith("chb24/")]
    core_seiz=[r for r in seizure_records if not r.startswith("chb24/")]
    provenance={"physionet_version":"1.0.0","records_current":len(records),"seizure_edf_current":len(seizure_records),
                "records_core_without_chb24":len(core),"seizure_edf_core_without_chb24":len(core_seiz),
                "selected_edf":len(set(selected)),"includes_chb24":any(r.startswith("chb24/") for r in selected)}
    return sorted(set(selected)),events,checksums,provenance

def resumable_download(rel,expected,retries=6):
    dest=CACHE/rel; part=dest.with_suffix(dest.suffix+".part"); dest.parent.mkdir(parents=True,exist_ok=True)
    if dest.exists() and expected and sha256_file(dest)==expected: return dest
    if dest.exists():
        if part.exists(): part.unlink()
        dest.replace(part)
    curl=shutil.which("curl.exe") or shutil.which("curl")
    if curl:
        for integrity_attempt in range(2):
            command=[curl,"--fail","--location","--continue-at","-","--retry",str(retries),
                     "--retry-delay","2","--retry-all-errors","--connect-timeout","30","--max-time","1800",
                     "--output",str(part),DATA_URL+rel]
            subprocess.run(command,check=True)
            if not expected or sha256_file(part)==expected:
                os.replace(part,dest); return dest
            # El parcial no puede reutilizarse: se elimina solo este archivo corrupto
            # y se realiza una descarga completa antes de declarar el fallo.
            part.unlink(missing_ok=True)
        raise RuntimeError(f"SHA256 no coincide después de descarga completa: {rel}")
    for attempt in range(retries):
        try:
            offset=part.stat().st_size if part.exists() else 0
            headers={"Range":f"bytes={offset}-"} if offset else {}
            req=urllib.request.Request(DATA_URL+rel,headers=headers)
            with urllib.request.urlopen(req,timeout=180) as response:
                append=offset>0 and getattr(response,"status",200)==206
                with part.open("ab" if append else "wb") as out:
                    shutil.copyfileobj(response,out,length=8*1024*1024)
            if expected and sha256_file(part)!=expected:
                part.unlink(missing_ok=True)
                raise IOError("SHA256 no coincide; se reiniciará únicamente este archivo")
            os.replace(part,dest); return dest
        except urllib.error.HTTPError as exc:
            if exc.code==416: part.unlink(missing_ok=True)
            if attempt+1==retries: raise RuntimeError(f"No se descargó {rel}: {exc}") from exc
            time.sleep(min(60,2**attempt))
        except Exception as exc:
            if attempt+1==retries: raise RuntimeError(f"No se descargó {rel}: {exc}") from exc
            time.sleep(min(60,2**attempt))

def ensure_downloads(selected,checksums):
    from concurrent.futures import ThreadPoolExecutor,as_completed
    pending=[]; partial=[]
    for rel in selected:
        p=CACHE/rel; part=p.with_suffix(p.suffix+".part"); expected=checksums.get(rel)
        if part.exists(): partial.append((rel,part.stat().st_size))
        if not p.exists() or not expected or sha256_file(p)!=expected: pending.append(rel)
    print(f"CHB-MIT: {len(selected)-len(pending)}/{len(selected)} EDF verificados; pendientes: {len(pending)}")
    if not pending:
        print("CHB-MIT ya está descargado y verificado; no es necesaria una nueva descarga.")
    else:
        for rel,size in partial: print(f"CHB-MIT parcial: {rel}; reanudación desde {size/(1024**2):.1f} MiB")
        print("CHB-MIT: descarga reanudable activada en",CACHE)
        with ThreadPoolExecutor(max_workers=DOWNLOAD_WORKERS) as pool:
            jobs={pool.submit(resumable_download,rel,checksums.get(rel)):rel for rel in pending}
            for i,fut in enumerate(as_completed(jobs),1):
                fut.result(); print(f"CHB-MIT descarga verificada {i}/{len(jobs)}: {jobs[fut]}")
    bad=[rel for rel in selected if not (CACHE/rel).exists() or sha256_file(CACHE/rel)!=checksums.get(rel)]
    if bad: raise RuntimeError(f"EDF sin integridad: {bad[:8]}")
    return {"selected":len(selected),"verified":len(selected),"downloaded":len(pending),"resumed":len(partial)}


In [ ]:
# BLOQUE 3 — construcción real y perfil exclusivo de desarrollo
def reservoir_add(store,metas,item,meta,seen,limit,rng):
    seen+=1
    if len(store)<limit: store.append(item); metas.append(meta)
    else:
        j=int(rng.integers(0,seen))
        if j<limit: store[j]=item; metas[j]=meta
    return seen

def build_real(scope,out_dir):
    selected,events,checksums,provenance=official_index(scope)
    ensure_downloads(selected,checksums)
    if out_dir.exists(): shutil.rmtree(out_dir)
    out_dir.mkdir(parents=True)
    persons=sorted(set(CASE_PERSON.get(r.split("/")[0],r.split("/")[0]) for r in selected))
    windows=[]; metadata=[]
    for person in persons:
        rng=np.random.default_rng(SEED+stable_int(person)); pos=[]; neg=[]; pm=[]; nm=[]; sp=sn=0
        files=[r for r in selected if CASE_PERSON.get(r.split("/")[0],r.split("/")[0])==person]
        for rel in files:
            raw=mne.io.read_raw_edf(CACHE/rel,preload=True,verbose="ERROR"); raw.pick("eeg")
            data=raw.get_data(); fs_in=float(raw.info["sfreq"])
            if fs_in!=FS: data=mne.filter.resample(data,down=fs_in/FS,npad="auto",axis=1)
            x=common_bipolar(data*1e6,raw.ch_names)
            if x is None:
                print("SKIP montaje incompleto",rel); del raw,data; gc.collect(); continue
            rel_events=events.get(rel,[]); control=len(rel_events)==0
            for s in range(0,len(x)-WIN_PTS+1,WIN_PTS):
                base={"subject_id":person,"case":rel.split("/")[0],"file":rel,"start_s":s/FS}
                if control:
                    sn=reservoir_add(neg,nm,x[s:s+WIN_PTS],{**base,"label":0,"event_id":"","ictal_fraction":0.0},sn,MAX_WINDOWS_PER_SUBJECT_CLASS,rng)
                else:
                    frac=sum(max(0,min((s+WIN_PTS)/FS,b)-max(s/FS,a)) for a,b in rel_events)/WIN_SEC
                    if frac>=0.5:
                        eid=next((j for j,(a,b) in enumerate(rel_events) if max(s/FS,a)<min((s+WIN_PTS)/FS,b)),0)
                        sp=reservoir_add(pos,pm,x[s:s+WIN_PTS],{**base,"label":1,"event_id":f"{rel}:{eid}","ictal_fraction":frac},sp,MAX_WINDOWS_PER_SUBJECT_CLASS,rng)
            del raw,data,x; gc.collect()
        if pos and neg:
            windows.extend(pos+neg); metadata.extend(pm+nm)
        print(person,"ictal",len(pos),"control",len(neg))
    if not windows: raise RuntimeError("No se construyeron ventanas reales")
    X=np.asarray(windows,dtype=np.float32); meta=pd.DataFrame(metadata); y=meta.label.to_numpy(np.int8); pid=meta.subject_id.to_numpy(str)
    np.save(out_dir/"X.npy",X); np.save(out_dir/"y.npy",y); np.save(out_dir/"pid.npy",pid); meta.to_csv(out_dir/"metadata.csv",index=False)
    artifacts={
        "X":{"sha256":sha256_file(out_dir/"X.npy"),"bytes":(out_dir/"X.npy").stat().st_size,"shape":list(X.shape),"dtype":str(X.dtype)},
        "y":{"sha256":sha256_file(out_dir/"y.npy"),"bytes":(out_dir/"y.npy").stat().st_size,"shape":list(y.shape),"dtype":str(y.dtype)},
        "pid":{"sha256":sha256_file(out_dir/"pid.npy"),"bytes":(out_dir/"pid.npy").stat().st_size,"shape":list(pid.shape),"dtype":str(pid.dtype)},
        "metadata":{"sha256":sha256_file(out_dir/"metadata.csv"),"bytes":(out_dir/"metadata.csv").stat().st_size,"rows":len(meta)},
    }
    manifest={"schema_version":"5.1-real","scope":scope,"fs_hz":FS,"window_s":WIN_SEC,"n_windows":len(y),
              "subjects":sorted(set(pid.tolist())),"source":provenance,"selected_files":selected,
              "selected_file_sha256":{r:checksums[r] for r in selected},"artifacts":artifacts,
              "created_utc":time.strftime("%Y-%m-%dT%H:%M:%SZ",time.gmtime())}
    (out_dir/"manifest.json").write_text(json.dumps(manifest,indent=2,ensure_ascii=False),encoding="utf-8")
    return X,y,pid,meta,manifest

def promote_named(work,mapping):
    backup=WORK_ROOT/"promotion_backup"; backup.mkdir(parents=True,exist_ok=True); moved=[]
    try:
        for src_name,dst in mapping.items():
            src=work/src_name
            if dst.exists():
                b=backup/dst.name; os.replace(dst,b); moved.append((b,dst))
            os.replace(src,dst)
    except Exception:
        for b,dst in reversed(moved):
            if dst.exists(): dst.unlink()
            os.replace(b,dst)
        raise
    shutil.rmtree(backup,ignore_errors=True)

def write_dev_profile(X,y,pid,meta,manifest):
    F=extract_features(X,zscore=False,tag="DEV_raw")
    rows=[]
    for subject in sorted(set(pid.tolist())):
        for cls in (0,1):
            z=F[(pid==subject)&(y==cls)]
            if not len(z): continue
            for j,name in enumerate(FEATURE_NAMES):
                rows.append({"subject_id":subject,"class":cls,"feature":name,"median":float(np.median(z[:,j])),
                             "iqr":float(stats.iqr(z[:,j])),"q05":float(np.percentile(z[:,j],5)),"q95":float(np.percentile(z[:,j],95)),"n":len(z)})
    df=pd.DataFrame(rows); tmp=DEV_WORK/"dev_feature_profile.csv"; df.to_csv(tmp,index=False)
    window_features=meta.reset_index(drop=True).copy()
    for j,name in enumerate(FEATURE_NAMES): window_features[name]=F[:,j]
    window_features.to_csv(DEV_WORK/"dev_window_features.csv",index=False)
    event_rows=[]
    ictal=window_features[(window_features.label==1)&window_features.event_id.notna()&(window_features.event_id.astype(str)!="")]
    for event_id,g in ictal.groupby("event_id"):
        g=g.sort_values("start_s"); q=max(1,len(g)//3); first=g.iloc[:q]; last=g.iloc[-q:]
        event_rows.append({"event_id":event_id,"subject_id":g.subject_id.iloc[0],"n_windows":len(g),
                           "freq_start_hz":float(first.dom_freq.median()),"freq_end_hz":float(last.dom_freq.median()),
                           "ptp_start_uv":float(first.ptp_med.median()),"ptp_end_uv":float(last.ptp_med.median()),
                           "frequency_decreased":bool(first.dom_freq.median()>last.dom_freq.median()),
                           "amplitude_increased":bool(last.ptp_med.median()>first.ptp_med.median())})
    pd.DataFrame(event_rows).to_csv(DEV_WORK/"dev_event_evolution.csv",index=False)
    source_contract={"physionet_version":manifest["source"]["physionet_version"],"scope":manifest["scope"],
                     "fs_hz":manifest["fs_hz"],"window_s":manifest["window_s"],"subjects":manifest["subjects"],
                     "selected_files":manifest["selected_files"],"selected_file_sha256":manifest["selected_file_sha256"],
                     "n_windows":manifest["n_windows"]}
    source_contract_sha256=hashlib.sha256(json.dumps(source_contract,sort_keys=True,separators=(",",":"),ensure_ascii=False).encode("utf-8")).hexdigest()
    profile={"schema_version":"5.1-dev","subjects":sorted(set(pid.tolist())),"n_windows":int(len(y)),
             "source_contract_sha256":source_contract_sha256,"feature_names":FEATURE_NAMES,
             "feature_domains":FEATURE_DOMAINS,"n_events_with_evolution":len(event_rows)}
    atomic_write_json(DEV_WORK/"development_profile.json",profile)
    promote_named(DEV_WORK,{"dev_feature_profile.csv":OUT_DIR/"dev_feature_profile.csv",
                            "dev_window_features.csv":OUT_DIR/"dev_window_features.csv",
                            "dev_event_evolution.csv":OUT_DIR/"dev_event_evolution.csv",
                            "development_profile.json":OUT_DIR/"development_profile.json",
                            "manifest.json":OUT_DIR/"dev_source_manifest.json",
                            "metadata.csv":OUT_DIR/"dev_window_metadata.csv"})
    shutil.rmtree(DEV_WORK,ignore_errors=True)
    print("DESARROLLO: SUPPORTED",profile)

REAL={"ready":False,"reason":"modo sin cohorte externa"}
if MODE=="prepare_dev":
    Xd,yd,pd_,md,mf=build_real("dev",DEV_WORK); write_dev_profile(Xd,yd,pd_,md,mf)
    record_stage("S1_development_profile","SUPPORTED",{
      "development_profile":OUT_DIR/"development_profile.json",
      "dev_source_manifest":OUT_DIR/"dev_source_manifest.json",
      "dev_feature_profile":OUT_DIR/"dev_feature_profile.csv"},
      {"subjects":sorted(DEV_CASES),"next_step":"execute 1_0 in full mode"})
elif MODE=="external":
    paths=[REAL_DIR/"X_real.npy",REAL_DIR/"y_real.npy",REAL_DIR/"pid_real.npy",REAL_DIR/"real_window_metadata.csv",REAL_DIR/"real_manifest.json"]
    if REBUILD_REAL or not all(p.exists() for p in paths):
        Xr,yr,pr,mr,mf=build_real("all",REAL_WORK)
        if len(set(pr.tolist()))<15: raise RuntimeError("cohorte externa insuficiente")
        promote_named(REAL_WORK,{"X.npy":paths[0],"y.npy":paths[1],"pid.npy":paths[2],"metadata.csv":paths[3],"manifest.json":paths[4]})
        shutil.rmtree(REAL_WORK,ignore_errors=True)
    Xr=np.load(paths[0],mmap_mode="r"); yr=np.load(paths[1]); pr=np.load(paths[2]); mr=pd.read_csv(paths[3]); rm=json.loads(paths[4].read_text())
    role_paths={"X":paths[0],"y":paths[1],"pid":paths[2],"metadata":paths[3]}
    if set(rm.get("artifacts",{}))!=set(role_paths): raise RuntimeError("manifiesto REAL sin hashes completos")
    bad=[role for role,p in role_paths.items() if sha256_file(p)!=rm["artifacts"][role].get("sha256")]
    if bad: raise RuntimeError(f"artefactos REAL con SHA256 inválido: {bad}")
    if Xr.shape!=(len(yr),int(FS*WIN_SEC),len(BIPOLAR)) or len(pr)!=len(yr) or len(mr)!=len(yr):
        raise RuntimeError("formas REAL incompatibles")
    if set(np.unique(yr).tolist())!={0,1}: raise RuntimeError("clases REAL incompletas")
    for start in range(0,len(Xr),1000):
        if not np.isfinite(np.asarray(Xr[start:start+1000])).all(): raise RuntimeError("NaN/Inf en REAL")
    if set(pr.tolist())!=set(mr.subject_id.astype(str)): raise RuntimeError("IDs REAL incompatibles")
    REAL={"ready":True,"X":Xr,"y":yr,"pid":pr,"meta":mr,"manifest":rm,"subjects":sorted(set(pr.tolist())),
          "manifest_sha256":sha256_file(paths[4])}
    record_stage("S4_chbmit_derived","SUPPORTED",{"real_manifest":paths[4],"metadata":paths[3]},
                 {"subjects":len(set(pr.tolist())),"windows":len(yr),"next_step":"execute siena_external"})
print("REAL:","SUPPORTED" if REAL["ready"] else "NOT EVALUABLE — "+REAL["reason"])

In [ ]:
# BLOQUE 4 — fidelidad, cobertura morfológica, diversidad y proximidad
def sample_idx(y,n,seed):
    rng=np.random.default_rng(seed); out=[]
    for cls in (0,1):
        a=np.flatnonzero(np.asarray(y)==cls); out.extend(rng.choice(a,min(n,len(a)),replace=False).tolist())
    rng.shuffle(out); return np.asarray(out,dtype=int)

def synthetic_samples(n=1200):
    Xs=[]; ys=[]; scenarios=[]; split_tags=[]
    for k in ("train","val","test"):
        idx=sample_idx(SYN["y"][k],n,SEED+len(Xs))
        Xs.append(synthetic_to_bipolar(np.asarray(SYN["X"][k][idx])))
        ys.append(SYN["y"][k][idx])
        scenarios.extend(SYN["meta"][k].iloc[idx].scenario.astype(str).tolist())
        split_tags.extend([k]*len(idx))
    X=np.concatenate(Xs); y=np.concatenate(ys)
    return X,y,np.asarray(scenarios),np.asarray(split_tags)

def internal_separability(Fs_raw,Fs_z,ys,split_tags):
    # Evaluación exclusivamente entre pacientes sintéticos disjuntos: TRAIN+VAL -> TEST.
    tr=np.flatnonzero(split_tags!="test"); te=np.flatnonzero(split_tags=="test"); rows=[]
    for mode,F in (("raw",Fs_raw),("zscore",Fs_z)):
        for model in ("LR","RF"):
            auc,ap=clf_scores(F[tr],ys[tr],F[te],ys[te],model,SEED)
            rows.append({"mode":mode,"model":model,"protocol":"SYN_TRAIN+VAL→SYN_TEST",
                         "roc_auc":auc,"average_precision":ap,"n_train":len(tr),"n_test":len(te),
                         "train_splits":"train,val","test_split":"test","patient_disjoint":True,
                         "status":"SUPPORTED" if np.isfinite(auc) else "NOT EVALUABLE"})
    return pd.DataFrame(rows)

def cliffs_delta(a,b):
    a=np.asarray(a); b=np.asarray(b); d=a[:,None]-b[None,:]
    return float((np.sum(d>0)-np.sum(d<0))/d.size)

def mmd2_rbf(X,Y):
    Z=np.vstack([X,Y]); D=cdist(Z,Z); pos=D[D>0]; gamma=1/(2*np.median(pos)**2+1e-12)
    Kx=np.exp(-gamma*cdist(X,X,"sqeuclidean")); Ky=np.exp(-gamma*cdist(Y,Y,"sqeuclidean")); Kxy=np.exp(-gamma*cdist(X,Y,"sqeuclidean")); np.fill_diagonal(Kx,0); np.fill_diagonal(Ky,0)
    return float(Kx.sum()/(len(X)*(len(X)-1))+Ky.sum()/(len(Y)*(len(Y)-1))-2*Kxy.mean())

def distribution_metrics(Fs,Fr,ys,yr,scenarios,pid):
    rows=[]; coverage=[]
    groups=[("interictal",ys==0,yr==0),("ictal_all",ys==1,yr==1),
            ("generalized_absence",(ys==1)&(scenarios=="generalized_absence"),yr==1),
            ("focal_temporal",(ys==1)&(scenarios=="focal_temporal"),yr==1)]
    rng=np.random.default_rng(SEED)
    for group,sm,rm in groups:
        si=np.flatnonzero(sm); ri=np.flatnonzero(rm); si=rng.choice(si,min(600,len(si)),False); ri=rng.choice(ri,min(600,len(ri)),False)
        for j,name in enumerate(FEATURE_NAMES):
            a=Fs[si,j]; b=Fr[ri,j]; q05s,q95s=np.percentile(a,[5,95]); q05r,q95r=np.percentile(b,[5,95])
            rows.append({"group":group,"feature":name,"wasserstein":float(stats.wasserstein_distance(a,b)),
                         "wasserstein_over_real_iqr":float(stats.wasserstein_distance(a,b)/(stats.iqr(b)+1e-12)),
                         "ks_stat":float(stats.ks_2samp(a,b).statistic),"cliffs_delta":cliffs_delta(a,b)})
            coverage.append({"group":group,"feature":name,"real_inside_syn_05_95":float(np.mean((b>=q05s)&(b<=q95s))),
                             "syn_inside_real_05_95":float(np.mean((a>=q05r)&(a<=q95r)))})
    subject_rows=[]
    for subject in sorted(set(pid.tolist())):
        for cls in (0,1):
            rr=np.flatnonzero((yr==cls)&(pid==subject)); ss=np.flatnonzero(ys==cls)
            if not len(rr) or not len(ss): continue
            for j,name in enumerate(FEATURE_NAMES):
                subject_rows.append({"subject":subject,"class":cls,"feature":name,
                    "wasserstein_over_real_iqr":float(stats.wasserstein_distance(Fs[ss,j],Fr[rr,j])/(stats.iqr(Fr[rr,j])+1e-12)),
                    "ks_stat":float(stats.ks_2samp(Fs[ss,j],Fr[rr,j]).statistic),"cliffs_delta":cliffs_delta(Fs[ss[:min(400,len(ss))],j],Fr[rr[:min(400,len(rr))],j])})
    return pd.DataFrame(rows),pd.DataFrame(coverage),pd.DataFrame(subject_rows)

def proximity_diversity(Fs,Fr,ys,yr):
    rng=np.random.default_rng(SEED); rows=[]; scaler=StandardScaler().fit(Fr)
    for cls in (0,1):
        si=rng.choice(np.flatnonzero(ys==cls),min(500,np.sum(ys==cls)),False); ri=rng.choice(np.flatnonzero(yr==cls),min(500,np.sum(yr==cls)),False)
        xs=scaler.transform(Fs[si]); xr=scaler.transform(Fr[ri]); rr=NearestNeighbors(n_neighbors=2).fit(xr).kneighbors(xr)[0][:,1]; sr=NearestNeighbors(n_neighbors=1).fit(xr).kneighbors(xs)[0][:,0]; ss=NearestNeighbors(n_neighbors=2).fit(xs).kneighbors(xs)[0][:,1]
        threshold=float(np.percentile(rr,5)); rows.append({"class":cls,"fraction_close_feature_space":float(np.mean(sr<threshold)),
            "syn_real_nn_median":float(np.median(sr)),"real_real_nn_median":float(np.median(rr)),"syn_syn_nn_median":float(np.median(ss)),
            "diversity_ratio_syn_real":float(np.median(ss)/(np.median(rr)+1e-12)),"mmd2_rbf":mmd2_rbf(xs,xr),"not_privacy":True})
    return pd.DataFrame(rows)

In [ ]:
# BLOQUE 5 — protocolos LOSO y bootstrap por persona
def balanced(y,n,seed):
    rng=np.random.default_rng(seed); out=[]
    for cls in (0,1):
        a=np.flatnonzero(np.asarray(y)==cls); out.extend(rng.choice(a,min(n,len(a)),False).tolist())
    rng.shuffle(out); return np.asarray(out,dtype=int)

def smote_training_only(X,y,seed):
    from imblearn.over_sampling import SMOTE
    counts=np.bincount(np.asarray(y,dtype=int),minlength=2)
    if counts.min()<2 or counts[0]==counts[1]: return np.asarray(X),np.asarray(y)
    return SMOTE(random_state=seed,k_neighbors=min(5,int(counts.min())-1)).fit_resample(np.asarray(X),np.asarray(y))

def clf_scores(Xtr,ytr,Xte,yte,name,seed):
    if len(np.unique(ytr))<2 or len(np.unique(yte))<2: return np.nan,np.nan
    model=(make_pipeline(StandardScaler(),LogisticRegression(max_iter=2000,class_weight="balanced",random_state=seed)) if name=="LR" else RandomForestClassifier(n_estimators=300,n_jobs=1,class_weight="balanced",random_state=seed))
    model.fit(Xtr,ytr); p=model.predict_proba(Xte)[:,1]
    return float(roc_auc_score(yte,p)),float(average_precision_score(yte,p))

def bootstrap_subject(values,seed=SEED,n_boot=2000):
    a=np.asarray(values,float); a=a[np.isfinite(a)]
    if not len(a): return np.nan,np.nan,np.nan
    rng=np.random.default_rng(seed); boot=np.asarray([a[rng.integers(0,len(a),len(a))].mean() for _ in range(n_boot)])
    return float(a.mean()),float(np.percentile(boot,2.5)),float(np.percentile(boot,97.5))

def run_protocols(Fs_raw,Fs_z,ys,Fr_raw,Fr_z,yr,pid):
    dev=set(DEV_CASES); external=sorted(set(pid.tolist())-dev); rows=[]
    if set(pid.tolist()) & dev: raise RuntimeError("los sujetos de desarrollo llegaron a run_protocols")
    for subject in external:
        test=np.flatnonzero(pid==subject); train=np.flatnonzero(pid!=subject); seed=SEED+stable_int(subject)%10000
        # R2R conserva la prevalencia observada. El límite es computacional y se muestrea
        # sin usar el sujeto de prueba. SMOTE se ajusta después y solo sobre este training.
        rng=np.random.default_rng(seed)
        train=rng.choice(train,min(4000,len(train)),replace=False)
        if len(np.unique(yr[train]))<2: raise RuntimeError(f"training LOSO monoclase: {subject}")
        syn_take=balanced(ys,3000,seed+1)
        for mode,Fs,Fr in (("raw",Fs_raw,Fr_raw),("zscore",Fs_z,Fr_z)):
            Xrtr,yrtr=Fr[train],yr[train]; Xrte,yrte=Fr[test],yr[test]; Xst,yst=Fs[syn_take],ys[syn_take]
            Xsm,ysm=smote_training_only(Xrtr,yrtr,seed)
            specs={"R2R":(Xrtr,yrtr,Xrte,yrte),"TSTR":(Xst,yst,Xrte,yrte),"TRTS":(Xrtr,yrtr,Fs,ys),
                   "R+S→R":(np.vstack([Xrtr,Xst]),np.concatenate([yrtr,yst]),Xrte,yrte),"SMOTE→R":(Xsm,ysm,Xrte,yrte)}
            for model in ("LR","RF"):
                for protocol,(a,b,c,d) in specs.items():
                    auc,ap=clf_scores(a,b,c,d,model,seed); rows.append({"subject":subject,"mode":mode,"model":model,"protocol":protocol,"roc_auc":auc,"average_precision":ap,"n_test":len(d)})
    detail=pd.DataFrame(rows); summaries=[]
    for keys,g in detail.groupby(["mode","model","protocol"]):
        for metric in ("roc_auc","average_precision"):
            mean,lo,hi=bootstrap_subject(g[metric]); summaries.append({"mode":keys[0],"model":keys[1],"protocol":keys[2],"metric":metric,"macro_mean":mean,"ci95_low":lo,"ci95_high":hi,"n_subjects":g.subject.nunique()})
    return detail,pd.DataFrame(summaries)

In [ ]:
# BLOQUE 6 — ejecución externa, estados y auditoría de afirmaciones
summary={"mode":MODE,"synthetic_status":"SUPPORTED" if SYN["ready"] else "NOT EVALUABLE",
         "real_status":"SUPPORTED" if REAL["ready"] else "NOT EVALUABLE","external_validation_status":"NOT EVALUABLE",
         "claims":{},"generated_utc":time.strftime("%Y-%m-%dT%H:%M:%SZ",time.gmtime())}

if MODE=="external" and SYN["ready"] and REAL["ready"]:
    Xs,ys,scenarios,split_tags=synthetic_samples()
    pid_all=np.asarray(REAL["pid"]); external_mask=~np.isin(pid_all,np.asarray(sorted(DEV_CASES)))
    Xr=np.asarray(REAL["X"])[external_mask]; yr=np.asarray(REAL["y"])[external_mask]; pid=pid_all[external_mask]
    if set(pid.tolist()) & set(DEV_CASES): raise RuntimeError("fuga de sujetos de desarrollo a evaluación externa")
    if len(set(pid.tolist()))!=18: raise RuntimeError(f"se esperaban 18 sujetos externos y hay {len(set(pid.tolist()))}")
    Fs_raw=extract_features(Xs,False,"SYN_raw"); Fs_z=extract_features(Xs,True,"SYN_zscore")
    Fr_raw=extract_features(Xr,False,"REAL_raw"); Fr_z=extract_features(Xr,True,"REAL_zscore")
    internal_df=internal_separability(Fs_raw,Fs_z,ys,split_tags)
    fidelity_df,coverage_df,subject_fidelity_df=distribution_metrics(Fs_raw,Fr_raw,ys,yr,scenarios,pid)
    proximity_df=proximity_diversity(Fs_raw,Fr_raw,ys,yr)
    protocol_df,protocol_summary_df=run_protocols(Fs_raw,Fs_z,ys,Fr_raw,Fr_z,yr,pid)
    internal_df.to_csv(OUT_DIR/"01_internal_separability.csv",index=False)
    fidelity_df.to_csv(OUT_DIR/"02_fidelity_featurewise.csv",index=False); coverage_df.to_csv(OUT_DIR/"02b_morphology_coverage.csv",index=False)
    subject_fidelity_df.to_csv(OUT_DIR/"02c_fidelity_by_subject.csv",index=False); proximity_df.to_csv(OUT_DIR/"04_feature_space_proximity_diversity.csv",index=False)
    protocol_df.to_csv(OUT_DIR/"03_cross_domain_loso.csv",index=False); protocol_summary_df.to_csv(OUT_DIR/"03b_cross_domain_bootstrap.csv",index=False)
    target=protocol_summary_df[(protocol_summary_df["mode"]=="zscore")&(protocol_summary_df.model=="RF")&(protocol_summary_df.protocol=="TSTR")&(protocol_summary_df.metric=="roc_auc")].iloc[0]
    tstr_status="SUPPORTED" if target.macro_mean>0.5 and target.ci95_low>0.5 else "NOT SUPPORTED"
    ictal_fid=fidelity_df[fidelity_df.group=="ictal_all"]
    large_ictal_effects=int((ictal_fid.cliffs_delta.abs()>=0.474).sum())
    high_ictal_ks=int((ictal_fid.ks_stat>=0.5).sum())
    fidelity_status="NOT SUPPORTED" if (large_ictal_effects or high_ictal_ks) else "SUPPORTED"
    low_coverage=int(((coverage_df.real_inside_syn_05_95<0.5)|(coverage_df.syn_inside_real_05_95<0.5)).sum())
    coverage_status="NOT SUPPORTED" if low_coverage else "SUPPORTED"
    low_diversity=int((proximity_df.diversity_ratio_syn_real<0.8).sum())
    diversity_status="NOT SUPPORTED" if low_diversity else "SUPPORTED"
    internal_primary=internal_df[(internal_df["mode"]=="zscore")&(internal_df.model=="RF")].iloc[0]
    summary["external_validation_status"]="SUPPORTED"
    summary["claims"]={
        "generator_integrity":{"status":"SUPPORTED","manifest_sha256":sha256_file(SYN_DIR/"generation_manifest.json")},
        "real_cohort_integrity":{"status":"SUPPORTED","manifest_sha256":REAL["manifest_sha256"],"n_windows_all":int(len(REAL["y"])),"n_windows_external":int(len(yr)),"n_external_subjects":int(len(set(pid.tolist()))),"development_subjects_excluded":True},
        "internal_separability":{"status":"SUPPORTED","primary_auc":float(internal_primary.roc_auc),"file":"01_internal_separability.csv","not_external_validity":True},
        "external_TSTR_RF_zscore":{"status":tstr_status,"macro_auc":float(target.macro_mean),"ci95":[float(target.ci95_low),float(target.ci95_high)],"n_subjects":int(target.n_subjects)},
        "fidelity_similarity":{"status":fidelity_status,"large_ictal_cliff_effects":large_ictal_effects,"ictal_features_ks_ge_0_5":high_ictal_ks,"file":"02_fidelity_featurewise.csv"},
        "external_morphology_coverage":{"status":coverage_status,"feature_pairs_with_any_coverage_below_0_5":low_coverage,"interpretation":"cobertura cuantitativa; no diagnóstico clínico","file":"02b_morphology_coverage.csv"},
        "feature_space_proximity":{"status":"SUPPORTED","not_privacy":True,"file":"04_feature_space_proximity_diversity.csv"},
        "diversity_vs_real":{"status":diversity_status,"classes_below_0_8":low_diversity,"file":"04_feature_space_proximity_diversity.csv"},
    }
    favorable_all=(tstr_status=="SUPPORTED" and fidelity_status=="SUPPORTED" and coverage_status=="SUPPORTED" and diversity_status=="SUPPORTED")
    framework_verdict="VIABLE" if favorable_all else "VIABLE CON LIMITACIONES"
    framework_reason=("El framework operacional es reproducible y transferencia, fidelidad, cobertura y diversidad satisfacen los criterios declarados." if favorable_all else
                      "El generador es reproducible y TSTR está respaldado, pero fidelidad, cobertura externa y diversidad muestran desplazamiento de dominio.")
    summary["framework_verdict"]={"status":framework_verdict,"reason":framework_reason,"not_clinical_equivalence":True}
    # La tesis no se modifica en esta fase y aún contiene cifras históricas incompatibles.
    summary["doctoral_verdict"]={"status":"NO VIABLE EN SU REDACCIÓN ACTUAL",
        "reason":"El manuscrito proporcionado conserva resultados históricos obsoletos; debe sustituirlos por la matriz 06 antes de defender las conclusiones.",
        "conditional_after_revision":framework_verdict,"not_clinical_equivalence":True}
    smote_primary=protocol_summary_df[(protocol_summary_df["mode"]=="zscore")&(protocol_summary_df.model=="RF")&(protocol_summary_df.protocol=="SMOTE→R")&(protocol_summary_df.metric=="roc_auc")].iloc[0]
    r2r_primary=protocol_summary_df[(protocol_summary_df["mode"]=="zscore")&(protocol_summary_df.model=="RF")&(protocol_summary_df.protocol=="R2R")&(protocol_summary_df.metric=="roc_auc")].iloc[0]
    prox0=proximity_df[proximity_df["class"]==0].iloc[0]; prox1=proximity_df[proximity_df["class"]==1].iloc[0]
    claims=pd.DataFrame([
        {"thesis_claim":"N=3000 sujetos sintéticos","status":"VIGENTE","new_evidence":"generation_manifest.json","required_action":"mantener"},
        {"thesis_claim":"CHB-MIT limitado a 5 sujetos/24 EDF","status":"OBSOLETO","new_evidence":"23 personas procesadas; 18 externas; 171 EDF seleccionados","required_action":"sustituir métodos y limitaciones"},
        {"thesis_claim":"ventanas REAL con 50% de solapamiento","status":"OBSOLETO","new_evidence":"2 s sin solapamiento","required_action":"sustituir"},
        {"thesis_claim":"ROC-AUC interno histórico 0.999","status":"OBSOLETO","new_evidence":f"RF zscore={internal_primary.roc_auc:.6f}","required_action":"sustituir por 01_internal_separability.csv"},
        {"thesis_claim":"TSTR histórico 0.613","status":"OBSOLETO","new_evidence":f"RF zscore={target.macro_mean:.6f}; IC95% [{target.ci95_low:.6f}, {target.ci95_high:.6f}]","required_action":"sustituir"},
        {"thesis_claim":"R2R histórico 0.466","status":"OBSOLETO","new_evidence":f"RF zscore={r2r_primary.macro_mean:.6f}; IC95% [{r2r_primary.ci95_low:.6f}, {r2r_primary.ci95_high:.6f}]","required_action":"sustituir"},
        {"thesis_claim":"SMOTE histórico 0.475 ± 0.098","status":"OBSOLETO","new_evidence":f"RF zscore={smote_primary.macro_mean:.6f}; IC95% [{smote_primary.ci95_low:.6f}, {smote_primary.ci95_high:.6f}]","required_action":"sustituir"},
        {"thesis_claim":"proximidad/memorización histórica 0.8%","status":"OBSOLETO","new_evidence":f"proximidad clase0={prox0.fraction_close_feature_space:.4f}; clase1={prox1.fraction_close_feature_space:.4f}","required_action":"renombrar; no afirmar privacidad"},
        {"thesis_claim":"diversidad histórica 0.569","status":"OBSOLETO","new_evidence":f"SYN/REAL clase0={prox0.diversity_ratio_syn_real:.4f}; clase1={prox1.diversity_ratio_syn_real:.4f}","required_action":"sustituir y declarar diversidad menor"},
        {"thesis_claim":"Gap/IQR global histórico 1.045 y rasgos históricos","status":"OBSOLETO","new_evidence":"02_fidelity_featurewise.csv","required_action":"sustituir cifras completas"},
        {"thesis_claim":"equivalencia clínica o capacidad diagnóstica","status":"NO RESPALDADO","new_evidence":"fuera del alcance","required_action":"no afirmar"},
        {"thesis_claim":"fidelidad uniforme SYN↔REAL","status":"NO RESPALDADO","new_evidence":f"{large_ictal_effects} efectos ictales Cliff grandes; {high_ictal_ks} KS≥0.5","required_action":"reportar desplazamiento de dominio"},
        {"thesis_claim":"generador paramétrico controlado para investigación","status":"VIGENTE","new_evidence":"integridad, reproducibilidad y contrato interno aprobados","required_action":"mantener con limitaciones externas"},
    ])
    claims.to_csv(OUT_DIR/"06_thesis_claim_matrix.csv",index=False)
elif not SYN["ready"]:
    summary["claims"]["generator_integrity"]={"status":"NOT EVALUABLE","reason":SYN["reason"]}
elif MODE=="prepare_dev":
    summary["claims"]["development_profile"]={"status":"SUPPORTED","file":"development_profile.json"}
else:
    summary["claims"]["external_validation"]={"status":"NOT EVALUABLE","reason":"ejecutar modo external después de congelar el generador"}

if MODE=="external":
    (OUT_DIR/"05_final_summary.json").write_text(json.dumps(summary,indent=2,ensure_ascii=False),encoding="utf-8")
    pd.DataFrame([{"claim":k,"status":v.get("status")} for k,v in summary["claims"].items()]).to_csv(OUT_DIR/"05_final_verdict_lines.csv",index=False)
    print(json.dumps(summary,indent=2,ensure_ascii=False))

In [ ]:
# BLOQUE 7 — validación multicorpus con Siena Scalp EEG
SIENA_CACHE = REAL_DIR / "siena_scalp_cache"
SIENA_BASE_URL = "https://physionet-open.s3.amazonaws.com/siena-scalp-eeg/1.0.0/"
SIENA_DOWNLOAD_WORKERS = max(1,int(os.environ.get("EEGSYN_SIENA_DOWNLOAD_WORKERS","8")))
SIENA_MAX_PER_SUBJECT_PHASE = 400
SIENA_REBUILD = os.environ.get("EEGSYN_REBUILD_SIENA", "0") == "1"
SIENA_REQUIRED = ["Fp1","Fp2","F7","F3","Fz","F4","F8","T3","C3","Cz","C4","T4","T5","P3","Pz","P4","T6","O1","O2"]

def siena_clock_seconds(value):
    m=re.search(r"(?<!\d)([0-2]?\d)[.:](\d{2})[.:](\d{2})(?!\d)",str(value))
    if not m: return None
    h,minute,sec=map(int,m.groups())
    return None if h>23 or minute>59 or sec>59 else h*3600+minute*60+sec

def parse_siena_annotation(path):
    text=path.read_text(encoding="utf-8",errors="replace")
    text=re.sub(r"(?<!\d)(\d)\s+(\d\.\d{2}\.\d{2})(?!\d)",r"\1\2",text)
    sampling=re.search(r"Data Sampling Rate:\s*([0-9.]+)",text,re.I)
    rows=[]; markers=list(re.finditer(r"Seizure\s+n\s*(\d+)",text,re.I))
    for i,mn in enumerate(markers):
        block=text[mn.start():(markers[i+1].start() if i+1<len(markers) else len(text))]; prefix=text[:mn.start()]
        mf=re.search(r"File name:\s*([^\r\n]+?\.edf)",block,re.I)
        mr=re.search(r"Registration start time:\s*([^\r\n]+)",block,re.I)
        if not mf:
            previous=re.findall(r"File name:\s*([^\r\n]+?\.edf)",prefix,re.I); mf=previous[-1] if previous else None
        if not mr:
            previous=re.findall(r"Registration start time:\s*([^\r\n]+)",prefix,re.I); mr=previous[-1] if previous else None
        ms=re.search(r"(?im)^(?:Seizure\s+)?Start time:\s*(.*?)(?=^(?:Seizure\s+)?End time:)",block,re.S|re.M)
        me=re.search(r"(?im)^(?:Seizure\s+)?End time:\s*([^\r\n]+)",block)
        if not (mf and mr and ms and me): continue
        times=re.findall(r"(?<!\d)([0-2]?\d[.:]\d{2}[.:]\d{2})(?!\d)",ms.group(1))
        if not times: continue
        chosen=times[-1] if "ELECTRIC ONSET" in ms.group(1).upper() else times[0]
        file_name=mf.group(1).strip() if hasattr(mf,"group") else str(mf).strip()
        registration=mr.group(1) if hasattr(mr,"group") else str(mr)
        rows.append({"event_number":int(mn.group(1)),"file":file_name,
          "registration_clock_s":siena_clock_seconds(registration),
          "start_clock_s":siena_clock_seconds(chosen),"end_clock_s":siena_clock_seconds(me.group(1)),
          "start_source":"electric" if "ELECTRIC ONSET" in ms.group(1).upper() else "annotated",
          "annotation_file":path.name})
    return rows,float(sampling.group(1)) if sampling else np.nan

def siena_channel_name(name):
    q=str(name).upper().replace("EEG","").replace("-REF","").replace("-LE","").replace("-AVG","")
    q=re.sub(r"[^A-Z0-9]","",q)
    q={"T7":"T3","P7":"T5","T8":"T4","P8":"T6"}.get(q,q)
    return "O1" if q=="1" else q

def siena_channel_indices(names):
    lookup={}
    for i,name in enumerate(names):
        q=siena_channel_name(name)
        if q and q not in lookup: lookup[q]=i
    missing=[c for c in SIENA_REQUIRED if c.upper() not in lookup]
    return (None,missing) if missing else ([lookup[c.upper()] for c in SIENA_REQUIRED],[])

def download_siena_official():
    SIENA_CACHE.mkdir(parents=True,exist_ok=True); sums=SIENA_CACHE/"SHA256SUMS.txt"
    if not sums.exists():
        print("Siena: descargando el índice oficial SHA256SUMS.txt")
        part=sums.with_suffix(sums.suffix+".part")
        with urllib.request.urlopen(SIENA_BASE_URL+"SHA256SUMS.txt",timeout=60) as src, part.open("wb") as dst: shutil.copyfileobj(src,dst,1024*1024)
        os.replace(part,sums)
    entries=[]
    for line in sums.read_text(encoding="utf-8",errors="replace").splitlines():
        m=re.match(r"([0-9a-fA-F]{64})\s+\*?(.+)",line.strip())
        if m: entries.append((m.group(2).replace("\\","/"),m.group(1).lower()))
    if len(entries)!=58 or sum(rel.lower().endswith(".edf") for rel,_ in entries)!=41:
        raise RuntimeError(f"Índice Siena inesperado: entradas={len(entries)}, EDF={sum(rel.lower().endswith('.edf') for rel,_ in entries)}")
    def fetch(item):
        rel,expected=item; final=SIENA_CACHE/Path(rel); part=Path(str(final)+".part"); final.parent.mkdir(parents=True,exist_ok=True)
        if final.exists() and sha256_file(final)==expected: return rel
        if part.exists() and sha256_file(part)==expected: os.replace(part,final); return rel
        for attempt in range(8):
            have=part.stat().st_size if part.exists() else 0; request=urllib.request.Request(SIENA_BASE_URL+rel,headers={"Range":f"bytes={have}-"} if have else {})
            try:
                with urllib.request.urlopen(request,timeout=120) as src:
                    resumed=(have>0 and getattr(src,"status",None)==206); mode="ab" if resumed else "wb"
                    with part.open(mode) as dst: shutil.copyfileobj(src,dst,4*1024*1024)
                if sha256_file(part)==expected: os.replace(part,final); return rel
                part.unlink(missing_ok=True)
            except urllib.error.HTTPError as exc:
                if exc.code==416: part.unlink(missing_ok=True)
                if attempt==7: raise
            except Exception:
                if attempt==7: raise
            time.sleep(min(30,2**attempt))
        raise RuntimeError(f"no se pudo descargar {rel}")
    pending=[]; partial=[]; verified_bytes=0
    for item in entries:
        rel,expected=item; final=SIENA_CACHE/Path(rel); part=Path(str(final)+".part")
        valid=final.exists() and sha256_file(final)==expected
        if valid: verified_bytes+=final.stat().st_size
        else: pending.append(item)
        if part.exists(): partial.append((rel,part.stat().st_size))
    print(f"Siena: {len(entries)-len(pending)}/{len(entries)} archivos verificados; EDF oficiales: 41; pendientes: {len(pending)}")
    if not pending:
        print(f"Siena ya está descargado y verificado ({verified_bytes/(1024**3):.2f} GiB); no es necesaria una nueva descarga.")
    else:
        free=shutil.disk_usage(SIENA_CACHE).free/(1024**3)
        if len(pending)==len(entries) and free<24:
            raise RuntimeError(f"Espacio insuficiente para Siena: {free:.2f} GiB libres; se requieren al menos 24 GiB")
        for rel,size in partial: print(f"Siena parcial: {rel}; reanudación desde {size/(1024**2):.1f} MiB")
        print("Siena: descarga reanudable activada en",SIENA_CACHE)
        with concurrent.futures.ThreadPoolExecutor(max_workers=SIENA_DOWNLOAD_WORKERS) as pool:
            for i,_ in enumerate(pool.map(fetch,pending),1): print(f"Siena descarga verificada {i}/{len(pending)}")
    return {"official_entries":len(entries),"official_edf":41,"downloaded":len(pending),"resumed":len(partial)}

def siena_official_index(verify_hashes=True):
    sums=SIENA_CACHE/"SHA256SUMS.txt"
    if not sums.exists(): raise FileNotFoundError("falta SHA256SUMS.txt de Siena")
    entries={}
    for line in sums.read_text(encoding="utf-8",errors="replace").splitlines():
        m=re.match(r"([0-9a-fA-F]{64})\s+\*?(.+)",line.strip())
        if m: entries[m.group(2).replace("\\","/")]=m.group(1).lower()
    bad=[rel for rel,expected in entries.items() if not (SIENA_CACHE/Path(rel)).exists() or (verify_hashes and sha256_file(SIENA_CACHE/Path(rel))!=expected)]
    if bad: raise RuntimeError(f"Siena incompleto o con SHA256 inválido: {bad[:8]}")
    subjects=pd.read_csv(SIENA_CACHE/"subject_info.csv"); subjects.columns=subjects.columns.astype(str).str.strip()
    edf_rel=sorted(rel for rel in entries if rel.lower().endswith(".edf"))
    if len(subjects)!=14 or len(edf_rel)!=41: raise RuntimeError(f"índice Siena inesperado: sujetos={len(subjects)}, EDF={len(edf_rel)}")
    return entries,subjects,edf_rel

def siena_event_relative(row):
    if None in (row["registration_clock_s"],row["start_clock_s"],row["end_clock_s"]): return None,None
    start=row["start_clock_s"]-row["registration_clock_s"]; end=row["end_clock_s"]-row["registration_clock_s"]
    if start<0: start+=86400
    if end<0: end+=86400
    if end<start: end+=86400
    return float(start),float(end)

def classify_siena_window(abs_start,abs_end,events):
    overlaps=[max(0.0,min(abs_end,e["abs_end"])-max(abs_start,e["abs_start"])) for e in events]
    overlap=max(overlaps or [0.0]); fraction=overlap/(abs_end-abs_start)
    event_id=events[int(np.argmax(overlaps))]["event_id"] if overlaps and overlap>0 else ""
    if fraction>=0.5: return "ictal",1,fraction,event_id
    if fraction>0: return "mixed",-2,fraction,event_id
    next_onset=min([e["abs_start"]-abs_end for e in events if e["abs_start"]>=abs_end] or [np.inf])
    prev_end=min([abs_start-e["abs_end"] for e in events if e["abs_end"]<=abs_start] or [np.inf])
    if 300<=next_onset<=1800: return "preictal",-1,0.0,""
    if 0<=next_onset<300: return "pre_onset_transition",-2,0.0,""
    if 0<=prev_end<1800: return "postictal",-2,0.0,""
    distances=[0.0 if abs_start<e["abs_end"] and abs_end>e["abs_start"] else min(abs(abs_start-e["abs_end"]),abs(e["abs_start"]-abs_end)) for e in events]
    return ("interictal_strict",0,0.0,"") if distances and min(distances)>=4*3600 else ("nonictal_control",-2,0.0,"")

def resampled_siena_window(raw,picks,start_sample,fs_in):
    pad=int(round(fs_in)); n=int(round(WIN_SEC*fs_in)); a=start_sample-pad; b=start_sample+n+pad
    if a<0 or b>raw.n_times: return None
    data=raw.get_data(picks=picks,start=a,stop=b)*1e6
    data=signal.resample(data,int(round(data.shape[1]*FS/fs_in)),axis=1)
    pad_out=int(round(pad*FS/fs_in)); data=data[:,pad_out:pad_out+WIN_PTS]
    if data.shape[1]!=WIN_PTS: return None
    idx={c.upper():i for i,c in enumerate(SIENA_REQUIRED)}
    return np.asarray(np.stack([data[idx[a.upper()]]-data[idx[b.upper()]] for _,a,b in BIPOLAR],axis=1),dtype=np.float32)

def build_siena_candidate(source_verified=False):
    entries,subjects,edf_rel=siena_official_index(verify_hashes=not source_verified); annotations={}; audit=[]; headers=[]; parsed_total=0
    for subject in subjects.patient_id.astype(str):
        annotations[subject]=parse_siena_annotation(SIENA_CACHE/subject/f"Seizures-list-{subject}.txt")[0]
        parsed_total += len(annotations[subject])
    if parsed_total != 47: raise RuntimeError(f"Siena annotation contract failed: expected 47 seizures, parsed {parsed_total}")
    for subject,rows in annotations.items():
        official_names=[Path(rel).name for rel in edf_rel if Path(rel).parts[0]==subject]
        for row in rows:
            row["file_original"]=row["file"]
            normalized_matches=[name for name in official_names if name.upper().replace("O","0")==row["file"].upper().replace("O","0")]
            if row["file"] in official_names: row["file_resolution"]="exact"
            elif len(normalized_matches)==1: row["file"]=normalized_matches[0]; row["file_resolution"]="documented_O_zero_filename_typo"
            elif len(official_names)==1: row["file"]=official_names[0]; row["file_resolution"]="single_edf_subject"
            else: row["file_resolution"]="unresolved"
    for rel in edf_rel:
        path=SIENA_CACHE/Path(rel); subject=Path(rel).parts[0]; fname=Path(rel).name
        raw=mne.io.read_raw_edf(path,preload=False,verbose="ERROR")
        picks,missing=siena_channel_indices(raw.ch_names); fs_in=float(raw.info["sfreq"])
        meas=raw.info.get("meas_date"); stamp=float(meas.timestamp()) if meas is not None else None
        duration=float(raw.n_times/fs_in); status="accepted" if picks is not None and stamp is not None else "excluded"
        reason="" if status=="accepted" else ("missing_channels:"+",".join(missing) if missing else "missing_measurement_date")
        headers.append({"subject_id":subject,"file":fname,"path":str(path),"fs_in":fs_in,"n_times":int(raw.n_times),
          "duration_s":duration,"timestamp":stamp,"picks":picks,"status":status,"reason":reason})
        for row in [r for r in annotations[subject] if r["file"]==fname]:
            start,end=siena_event_relative(row); accepted=(status=="accepted" and start is not None and 0<=start<end<=duration+1.0)
            audit.append({**row,"subject_id":subject,"start_relative_s":start,"end_relative_s":end,
              "recording_duration_s":duration,"accepted":bool(accepted),
              "reason":"" if accepted else ("invalid_or_outside_recording" if status=="accepted" else reason)})
        raw.close()
    if len(audit)!=parsed_total: raise RuntimeError(f"Siena audit contract failed: parsed {parsed_total}, audited {len(audit)}")
    audit_df=pd.DataFrame(audit); header_lookup={(h["subject_id"],h["file"]):h for h in headers}; accepted_events=[]
    for row in audit_df[audit_df.accepted].to_dict("records"):
        h=header_lookup[(row["subject_id"],row["file"])]
        accepted_events.append({"subject_id":row["subject_id"],"file":row["file"],
          "event_id":f'{row["subject_id"]}_{int(row["event_number"]):02d}',
          "abs_start":h["timestamp"]+row["start_relative_s"],"abs_end":h["timestamp"]+row["end_relative_s"]})
    candidate_by_subject={s:[] for s in subjects.patient_id.astype(str)}
    for h in headers:
        if h["status"]!="accepted": continue
        events=[e for e in accepted_events if e["subject_id"]==h["subject_id"]]
        first=int(np.ceil(h["fs_in"])); last=int(h["n_times"]-np.ceil((WIN_SEC+1)*h["fs_in"]))
        step=max(1,int(round(WIN_SEC*h["fs_in"])))
        for start in range(first,last+1,step):
            abs_start=h["timestamp"]+start/h["fs_in"]; phase,label,fraction,event_id=classify_siena_window(abs_start,abs_start+WIN_SEC,events)
            if phase in {"ictal","preictal","interictal_strict"}:
                candidate_by_subject[h["subject_id"]].append({"subject_id":h["subject_id"],"file":h["file"],
                  "start_sample":start,"start_s":start/h["fs_in"],"phase":phase,"label":label,
                  "ictal_fraction":fraction,"event_id":event_id})
    selected=[]
    for subject,candidates in candidate_by_subject.items():
        rng=np.random.default_rng(SEED+stable_int("siena_"+subject))
        for phase in ("ictal","preictal","interictal_strict"):
            group=[x for x in candidates if x["phase"]==phase]
            if len(group)>SIENA_MAX_PER_SUBJECT_PHASE:
                group=[group[i] for i in np.sort(rng.choice(len(group),SIENA_MAX_PER_SUBJECT_PHASE,replace=False))]
            selected.extend(group)
    windows=[]; kept=[]; selected_df=pd.DataFrame(selected)
    if selected_df.empty: raise RuntimeError("Siena no produjo candidatos")
    for (subject,fname),group in selected_df.groupby(["subject_id","file"],sort=True):
        h=header_lookup[(subject,fname)]; raw=mne.io.read_raw_edf(h["path"],preload=False,verbose="ERROR")
        for row in group.sort_values("start_sample").to_dict("records"):
            w=resampled_siena_window(raw,h["picks"],int(row["start_sample"]),h["fs_in"])
            if w is not None and np.isfinite(w).all(): windows.append(w); kept.append(row)
        raw.close(); gc.collect(); print("Siena",subject,fname,"ventanas",len(group))
    if not windows: raise RuntimeError("no se construyeron ventanas Siena")
    meta=pd.DataFrame(kept).merge(subjects,left_on="subject_id",right_on="patient_id",how="left")
    X=np.asarray(windows,dtype=np.float32); y=meta.label.to_numpy(np.int8); pid=meta.subject_id.to_numpy(str)
    work=WORK_ROOT/"siena_candidate"
    if work.exists(): shutil.rmtree(work)
    work.mkdir(parents=True)
    np.save(work/"siena_X.npy",X); np.save(work/"siena_y.npy",y); np.save(work/"siena_pid.npy",pid)
    meta.to_csv(work/"siena_window_metadata.csv",index=False); audit_df.to_csv(work/"siena_annotation_audit.csv",index=False)
    artifacts={}
    for name in ("siena_X.npy","siena_y.npy","siena_pid.npy","siena_window_metadata.csv","siena_annotation_audit.csv"):
        p=work/name; artifacts[name]={"sha256":sha256_file(p),"bytes":p.stat().st_size}
    manifest={"schema_version":"1.0-siena-multicorpus","source":{"name":"Siena Scalp EEG Database","version":"1.0.0",
      "doi":"10.13026/5d4a-j060","license":"CC BY 4.0","declared_subjects":14,"declared_edf":41,"declared_seizures":47},
      "fs_hz":FS,"window_s":WIN_SEC,"phase_definition":{"ictal":"occupancy >= 0.5","preictal":"30 to 5 min before onset",
      "interictal_strict":">= 4 h from any accepted seizure","mixed_transition_postictal":"excluded"},
      "prospective_acceptance_criteria":{"TSTR":"all SYN ictal scenarios; RF zscore macro AUROC > 0.5, bootstrap lower 95% > 0.5, at least 10 evaluable subjects",
      "complete_focal_morphology":"all predefined features inside the Siena REAL-REAL LOSO q95 envelope"},
      "n_subjects":int(meta.subject_id.nunique()),"n_windows":int(len(meta)),
      "phase_counts":{str(k):int(v) for k,v in meta.phase.value_counts().items()},
      "accepted_seizures":int(audit_df.accepted.sum()),"excluded_seizures":int((~audit_df.accepted).sum()),
      "official_sha256_entries":len(entries),"artifacts":artifacts,
      "synthetic_manifest_sha256":sha256_file(SYN_DIR/"generation_manifest.json"),
      "validator_notebook_sha256":sha256_file(PROJECT_ROOT/"1_1_EEGSynthesizer_VALIDATION.ipynb"),
      "software":{"python":sys.version,"numpy":np.__version__,"pandas":pd.__version__,"scipy":__import__("scipy").__version__,"mne":mne.__version__},
      "checkpoint_commit":"b554de3a2b5149d662c9e133d8d27320897eaee4",
      "created_utc":time.strftime("%Y-%m-%dT%H:%M:%SZ",time.gmtime())}
    (work/"siena_manifest.json").write_text(json.dumps(manifest,indent=2,ensure_ascii=False),encoding="utf-8")
    mapping={name:REAL_DIR/name for name in artifacts}; mapping["siena_manifest.json"]=REAL_DIR/"siena_manifest.json"
    promote_named(work,mapping); shutil.rmtree(work,ignore_errors=True)
    return manifest

def load_siena():
    names=["siena_X.npy","siena_y.npy","siena_pid.npy","siena_window_metadata.csv","siena_annotation_audit.csv"]
    mp=REAL_DIR/"siena_manifest.json"
    raw_status=download_siena_official()
    derived_complete=mp.exists() and all((REAL_DIR/n).exists() for n in names)
    if SIENA_REBUILD or not derived_complete:
        print("Siena derivados: construcción verificable activada")
        build_siena_candidate(source_verified=True)
    else:
        print("Siena derivados presentes; verificando manifiesto y hashes.")
    manifest=json.loads(mp.read_text(encoding="utf-8"))
    bad=[n for n in names if n not in manifest.get("artifacts",{}) or sha256_file(REAL_DIR/n)!=manifest["artifacts"][n]["sha256"]]
    if bad: raise RuntimeError(f"artefactos Siena inválidos: {bad}")
    print("Siena derivados: hashes válidos; no es necesario reconstruirlos.")
    X=np.load(REAL_DIR/"siena_X.npy",mmap_mode="r"); y=np.load(REAL_DIR/"siena_y.npy"); pid=np.load(REAL_DIR/"siena_pid.npy")
    meta=pd.read_csv(REAL_DIR/"siena_window_metadata.csv"); meta.columns=meta.columns.astype(str).str.strip()
    if X.shape!=(len(y),WIN_PTS,len(BIPOLAR)) or len(pid)!=len(y) or len(meta)!=len(y): raise RuntimeError("formas Siena incompatibles")
    for start in range(0,len(X),1000):
        if not np.isfinite(np.asarray(X[start:start+1000])).all(): raise RuntimeError("NaN/Inf en Siena")
    return {"X":X,"y":y,"pid":pid,"meta":meta,"manifest":manifest,"raw_status":raw_status}

def siena_distribution_metrics(Fs,Fr,ys,yr,scenarios,siena_meta):
    rows=[]; coverage=[]; rng=np.random.default_rng(SEED)
    focal_real=(yr==1)&(siena_meta.localization.astype(str).str.upper().to_numpy()=="T")
    groups=[("interictal",ys==0,yr==0),("ictal_all",ys==1,yr==1),
            ("focal_temporal",(ys==1)&(scenarios=="focal_temporal"),focal_real)]
    for group,sm,rm in groups:
        si=np.flatnonzero(sm); ri=np.flatnonzero(rm)
        si=rng.choice(si,min(600,len(si)),False); ri=rng.choice(ri,min(600,len(ri)),False)
        for j,name in enumerate(FEATURE_NAMES):
            a=Fs[si,j]; b=Fr[ri,j]; q05s,q95s=np.percentile(a,[5,95]); q05r,q95r=np.percentile(b,[5,95])
            rows.append({"group":group,"feature":name,"wasserstein":float(stats.wasserstein_distance(a,b)),
              "wasserstein_over_real_iqr":float(stats.wasserstein_distance(a,b)/(stats.iqr(b)+1e-12)),
              "ks_stat":float(stats.ks_2samp(a,b).statistic),"cliffs_delta":cliffs_delta(a,b)})
            coverage.append({"group":group,"feature":name,"real_inside_syn_05_95":float(np.mean((b>=q05s)&(b<=q95s))),
              "syn_inside_real_05_95":float(np.mean((a>=q05r)&(a<=q95r)))})
    return pd.DataFrame(rows),pd.DataFrame(coverage)

def real_real_reference(Fs,ys,Fsi,ysi,pid,Fchb,ychb):
    rows=[]
    for subject in sorted(set(pid.tolist())):
        test=pid==subject; train=pid!=subject
        for cls in (0,1):
            te=np.flatnonzero(test&(ysi==cls)); tr=np.flatnonzero(train&(ysi==cls))
            sy=np.flatnonzero(ys==cls); ch=np.flatnonzero(ychb==cls)
            if not len(te) or not len(tr) or not len(sy) or not len(ch): continue
            for j,name in enumerate(FEATURE_NAMES):
                denom=stats.iqr(Fsi[te,j])+1e-12
                rows.extend([
                  {"subject":subject,"class":cls,"feature":name,"comparison":"SIENA_LOSO_REAL_REAL",
                   "wasserstein_over_test_iqr":float(stats.wasserstein_distance(Fsi[tr,j],Fsi[te,j])/denom)},
                  {"subject":subject,"class":cls,"feature":name,"comparison":"CHB_TO_SIENA_REAL_REAL",
                   "wasserstein_over_test_iqr":float(stats.wasserstein_distance(Fchb[ch,j],Fsi[te,j])/denom)},
                  {"subject":subject,"class":cls,"feature":name,"comparison":"SYN_TO_SIENA",
                   "wasserstein_over_test_iqr":float(stats.wasserstein_distance(Fs[sy,j],Fsi[te,j])/denom)}])
    detail=pd.DataFrame(rows); summary=[]
    for (cls,name),g in detail.groupby(["class","feature"]):
        rr=g[g.comparison=="SIENA_LOSO_REAL_REAL"].wasserstein_over_test_iqr
        sr=g[g.comparison=="SYN_TO_SIENA"].wasserstein_over_test_iqr
        cr=g[g.comparison=="CHB_TO_SIENA_REAL_REAL"].wasserstein_over_test_iqr
        envelope=float(np.percentile(rr,95))
        summary.append({"class":cls,"feature":name,"siena_real_real_median":float(np.median(rr)),
          "siena_real_real_q95":envelope,"chb_to_siena_median":float(np.median(cr)),
          "syn_to_siena_median":float(np.median(sr)),"syn_within_real_real_q95":bool(np.median(sr)<=envelope),
          "chb_within_real_real_q95":bool(np.median(cr)<=envelope)})
    return detail,pd.DataFrame(summary)

def siena_mmd_by_subject(Fs,ys,Fsi,ysi,pid):
    rows=[]
    for subject in sorted(set(pid.tolist())):
        rng=np.random.default_rng(SEED+stable_int("mmd_"+subject))
        for cls in (0,1):
            te=np.flatnonzero((pid==subject)&(ysi==cls)); tr=np.flatnonzero((pid!=subject)&(ysi==cls)); sy=np.flatnonzero(ys==cls)
            if min(len(te),len(tr),len(sy))<2: continue
            te=rng.choice(te,min(250,len(te)),False); tr=rng.choice(tr,min(250,len(tr)),False); sy=rng.choice(sy,min(250,len(sy)),False)
            scaler=StandardScaler().fit(Fsi[tr]); a=scaler.transform(Fsi[te]); b=scaler.transform(Fsi[tr]); c=scaler.transform(Fs[sy])
            rr=mmd2_rbf(a,b); sr=mmd2_rbf(a,c)
            rows.append({"subject":subject,"class":cls,"mmd2_real_real":rr,"mmd2_syn_real":sr,
              "mmd_difference_syn_minus_real":float(sr-rr),"n_test":len(te),"n_reference":len(tr),"n_syn":len(sy)})
    return pd.DataFrame(rows)

def run_siena_protocols(Fs_raw,Fs_z,ys,scenarios,Fsi_raw,Fsi_z,ysi,pid,Fchb_raw,Fchb_z,ychb):
    rows=[]
    for subject in sorted(set(pid.tolist())):
        test=np.flatnonzero(pid==subject); train=np.flatnonzero(pid!=subject)
        seed=SEED+stable_int("siena_"+subject)%10000
        syn_take=balanced(ys,2500,seed); chb_take=balanced(ychb,2500,seed+1)
        focal_pool=np.flatnonzero((ys==0)|((ys==1)&(scenarios=="focal_temporal")))
        focal_take=focal_pool[balanced(ys[focal_pool],2500,seed+2)]
        for mode,Fs,Fsi,Fchb in (("raw",Fs_raw,Fsi_raw,Fchb_raw),("zscore",Fs_z,Fsi_z,Fchb_z)):
            specs={"SIENA_R2R_LOSO":(Fsi[train],ysi[train],Fsi[test],ysi[test]),
              "TSTR_SYN_TO_SIENA":(Fs[syn_take],ys[syn_take],Fsi[test],ysi[test]),
              "TSTR_FOCAL_SYN_TO_SIENA":(Fs[focal_take],ys[focal_take],Fsi[test],ysi[test]),
              "CHB_TO_SIENA_REAL_REAL":(Fchb[chb_take],ychb[chb_take],Fsi[test],ysi[test]),
              "CHB_PLUS_SYN_TO_SIENA":(np.vstack([Fchb[chb_take],Fs[syn_take]]),np.concatenate([ychb[chb_take],ys[syn_take]]),Fsi[test],ysi[test]),
              "SIENA_PLUS_SYN_LOSO":(np.vstack([Fsi[train],Fs[syn_take]]),np.concatenate([ysi[train],ys[syn_take]]),Fsi[test],ysi[test])}
            for model in ("LR","RF"):
                for protocol,(a,b,c,d) in specs.items():
                    auc,ap=clf_scores(a,b,c,d,model,seed)
                    rows.append({"subject":subject,"mode":mode,"model":model,"protocol":protocol,
                      "roc_auc":auc,"average_precision":ap,"n_test":len(d)})
    detail=pd.DataFrame(rows); summary=[]
    for keys,g in detail.groupby(["mode","model","protocol"]):
        for metric in ("roc_auc","average_precision"):
            valid=g[np.isfinite(g[metric])].copy(); mean,lo,hi=bootstrap_subject(valid[metric])
            summary.append({"mode":keys[0],"model":keys[1],"protocol":keys[2],"metric":metric,
              "macro_mean":mean,"ci95_low":lo,"ci95_high":hi,"n_subjects":valid.subject.nunique(),"n_subjects_total":g.subject.nunique()})
    return detail,pd.DataFrame(summary)

def siena_phase_metrics(F,meta):
    rows=[]
    for comparison,a_phase,b_phase in (("PREICTAL_VS_INTERICTAL","preictal","interictal_strict"),
                                        ("PREICTAL_VS_ICTAL","preictal","ictal")):
        a=np.flatnonzero(meta.phase.astype(str).to_numpy()==a_phase); b=np.flatnonzero(meta.phase.astype(str).to_numpy()==b_phase)
        if not len(a) or not len(b): continue
        for j,name in enumerate(FEATURE_NAMES):
            rows.append({"comparison":comparison,"feature":name,
              "wasserstein":float(stats.wasserstein_distance(F[a,j],F[b,j])),
              "ks_stat":float(stats.ks_2samp(F[a,j],F[b,j]).statistic),
              "cliffs_delta":cliffs_delta(F[a[:min(600,len(a))],j],F[b[:min(600,len(b))],j])})
    return pd.DataFrame(rows)

def siena_laterality_audit(X,y,pid,meta):
    rows=[]
    for subject in sorted(set(pid.tolist())):
        idx=np.flatnonzero((pid==subject)&(y==1))
        if not len(idx): continue
        scores=[]
        for w in np.asarray(X)[idx]:
            ptp=np.ptp(w,axis=0); left=ptp[:8].sum(); right=ptp[8:16].sum()
            scores.append((left-right)/(left+right+1e-12))
        truth=str(meta.loc[meta.subject_id.astype(str)==subject,"lateralization"].iloc[0]).upper()
        pred="L" if np.median(scores)>0 else "R"; evaluable=truth in {"L","R"}
        rows.append({"subject":subject,"true_lateralization":truth,"predicted_lateralization":pred,
          "median_left_minus_right_score":float(np.median(scores)),"evaluable":evaluable,
          "correct":bool(pred==truth) if evaluable else np.nan})
    return pd.DataFrame(rows)

if MODE=="siena_external":
    prerequisites=[SYN_DIR/"generation_manifest.json",REAL_DIR/"X_real.npy",REAL_DIR/"y_real.npy",REAL_DIR/"pid_real.npy",OUT_DIR/"05_final_summary.json"]
    missing=[str(p.relative_to(PROJECT_ROOT)) for p in prerequisites if not p.exists()]
    if missing:
        raise RuntimeError("Faltan prerrequisitos de Siena: "+", ".join(missing)+". Ejecute prepare_dev, 1_0 full y external, en ese orden.")
    if not SYN["ready"]: raise RuntimeError("dataset sintético no está congelado o íntegro")
    SIENA=load_siena(); primary=np.isin(SIENA["y"],[0,1])
    Xsi=np.asarray(SIENA["X"])[primary]; ysi=np.asarray(SIENA["y"])[primary]
    pidsi=np.asarray(SIENA["pid"])[primary]; msi=SIENA["meta"].loc[primary].reset_index(drop=True)
    if set(np.unique(ysi).tolist())!={0,1} or len(set(pidsi.tolist()))<10: raise RuntimeError("cohorte Siena primaria insuficiente")
    Xs,ys,scenarios,split_tags=synthetic_samples()
    Fs_raw=extract_features(Xs,False,"SYN_SIENA_raw"); Fs_z=extract_features(Xs,True,"SYN_SIENA_zscore")
    Fsi_raw=extract_features(Xsi,False,"SIENA_raw"); Fsi_z=extract_features(Xsi,True,"SIENA_zscore")
    Xchb=np.load(REAL_DIR/"X_real.npy",mmap_mode="r"); ychb_all=np.load(REAL_DIR/"y_real.npy"); pidchb=np.load(REAL_DIR/"pid_real.npy")
    chb_mask=~np.isin(pidchb,np.asarray(sorted(DEV_CASES)))
    Xchb_ext=np.asarray(Xchb)[chb_mask]; ychb=ychb_all[chb_mask]
    Fchb_raw=extract_features(Xchb_ext,False,"CHB_external_raw"); Fchb_z=extract_features(Xchb_ext,True,"CHB_external_zscore")
    fidelity,coverage=siena_distribution_metrics(Fs_raw,Fsi_raw,ys,ysi,scenarios,msi)
    proximity=proximity_diversity(Fs_raw,Fsi_raw,ys,ysi)
    rr_detail,rr_summary=real_real_reference(Fs_raw,ys,Fsi_raw,ysi,pidsi,Fchb_raw,ychb)
    mmd_subject=siena_mmd_by_subject(Fs_raw,ys,Fsi_raw,ysi,pidsi)
    protocol,protocol_summary=run_siena_protocols(Fs_raw,Fs_z,ys,scenarios,Fsi_raw,Fsi_z,ysi,pidsi,Fchb_raw,Fchb_z,ychb)
    Fsi_all=extract_features(np.asarray(SIENA["X"]),False,"SIENA_all_phases")
    phase_summary=siena_phase_metrics(Fsi_all,SIENA["meta"])
    laterality=siena_laterality_audit(Xsi,ysi,pidsi,msi)
    pd.read_csv(REAL_DIR/"siena_annotation_audit.csv").to_csv(OUT_DIR/"08_siena_annotation_audit.csv",index=False)
    fidelity.to_csv(OUT_DIR/"09_siena_fidelity_featurewise.csv",index=False)
    coverage.to_csv(OUT_DIR/"09_siena_morphology_coverage.csv",index=False)
    proximity.to_csv(OUT_DIR/"09_siena_proximity_diversity.csv",index=False)
    rr_detail.to_csv(OUT_DIR/"10_siena_real_real_by_subject.csv",index=False)
    rr_summary.to_csv(OUT_DIR/"10_siena_real_real_reference.csv",index=False)
    mmd_subject.to_csv(OUT_DIR/"10_siena_mmd_by_subject.csv",index=False)
    protocol.to_csv(OUT_DIR/"11_siena_transfer_loso.csv",index=False)
    protocol_summary.to_csv(OUT_DIR/"11_siena_bootstrap_summary.csv",index=False)
    phase_summary.to_csv(OUT_DIR/"12_siena_phase_summary.csv",index=False)
    laterality.to_csv(OUT_DIR/"12_siena_laterality_audit.csv",index=False)
    target=protocol_summary[(protocol_summary["mode"]=="zscore")&(protocol_summary.model=="RF")&
                            (protocol_summary.protocol=="TSTR_SYN_TO_SIENA")&(protocol_summary.metric=="roc_auc")].iloc[0]
    tstr_status=("NOT EVALUABLE" if target.n_subjects<10 else
                 ("SUPPORTED" if target.macro_mean>0.5 and target.ci95_low>0.5 else "NOT SUPPORTED"))
    selected_protocols={}
    for protocol_name in ("SIENA_R2R_LOSO","TSTR_SYN_TO_SIENA","TSTR_FOCAL_SYN_TO_SIENA","CHB_TO_SIENA_REAL_REAL","CHB_PLUS_SYN_TO_SIENA","SIENA_PLUS_SYN_LOSO"):
        row=protocol_summary[(protocol_summary["mode"]=="zscore")&(protocol_summary.model=="RF")&
                             (protocol_summary.protocol==protocol_name)&(protocol_summary.metric=="roc_auc")].iloc[0]
        selected_protocols[protocol_name]={"macro_auc":float(row.macro_mean),"ci95_low":float(row.ci95_low),"ci95_high":float(row.ci95_high),"n_subjects":int(row.n_subjects),"n_subjects_total":int(row.n_subjects_total)}
    focal=fidelity[fidelity.group=="focal_temporal"]
    focal_large=int((focal.cliffs_delta.abs()>=0.474).sum()); focal_ks=int((focal.ks_stat>=0.5).sum())
    rr_ictal=rr_summary[rr_summary["class"]==1]; within_fraction=float(rr_ictal.syn_within_real_real_q95.mean())
    lat_eval=laterality[laterality.evaluable==True]; lat_accuracy=float(lat_eval.correct.mean()) if len(lat_eval) else np.nan
    chb_summary=json.loads((OUT_DIR/"05_final_summary.json").read_text(encoding="utf-8"))
    chb_tstr=chb_summary.get("claims",{}).get("external_TSTR_RF_zscore",{})
    focal_status="SUPPORTED" if np.isclose(within_fraction,1.0) else "NOT SUPPORTED"
    verdict="VIABLE REFORZADO CON LIMITACIONES" if tstr_status=="SUPPORTED" else "VIABLE CON LIMITACIONES"
    reason=("La transferencia SYN→REAL se reprodujo en un segundo hospital adulto; la fidelidad y diversidad se interpretan frente a la referencia REAL–REAL."
            if tstr_status=="SUPPORTED" else
            ("La cohorte no aportó al menos 10 sujetos evaluables con ambas clases; la transferencia SYN→Siena no se interpreta."
             if tstr_status=="NOT EVALUABLE" else
             "La integridad multicorpus está respaldada, pero la transferencia SYN→Siena no superó el criterio prospectivo."))
    final={"mode":"siena_external","generated_utc":time.strftime("%Y-%m-%dT%H:%M:%SZ",time.gmtime()),
      "validation":{"notebook_sha256":sha256_file(PROJECT_ROOT/"1_1_EEGSynthesizer_VALIDATION.ipynb"),
        "siena_artifact_builder_notebook_sha256":SIENA["manifest"].get("validator_notebook_sha256")},
      "generator":{"status":"SUPPORTED","manifest_sha256":sha256_file(SYN_DIR/"generation_manifest.json"),"modified":False},
      "siena_integrity":{"status":"SUPPORTED","subjects":int(SIENA["meta"].subject_id.nunique()),
        "declared_seizures":47,"accepted_seizures":int(SIENA["manifest"]["accepted_seizures"]),
        "excluded_seizures":int(SIENA["manifest"]["excluded_seizures"]),
        "primary_windows":int(primary.sum()),"preictal_windows":int((SIENA["y"]==-1).sum())},
      "chb_mit_existing":{"status":chb_summary.get("framework_verdict",{}).get("status"),
        "tstr_macro_auc":chb_tstr.get("macro_auc"),"tstr_ci95":chb_tstr.get("ci95")},
      "siena_TSTR_RF_zscore":{"status":tstr_status,"macro_auc":float(target.macro_mean),
        "ci95":[float(target.ci95_low),float(target.ci95_high)],"n_subjects":int(target.n_subjects)},
      "siena_protocols_RF_zscore":selected_protocols,
      "focal_temporal_external":{"status":focal_status,
        "features_within_real_real_q95_fraction":within_fraction,"large_cliff_effects":focal_large,
        "features_ks_ge_0_5":focal_ks,"laterality_accuracy":lat_accuracy,
        "median_subject_mmd_difference_syn_minus_real":float(mmd_subject[mmd_subject["class"]==1].mmd_difference_syn_minus_real.median())},
      "diversity_vs_siena":{"status":"REPORTED","class_rows":proximity.to_dict("records")},
      "generalized_absence_external":{"status":"NOT EVALUABLE","reason":"Siena no contiene una cohorte de ausencias generalizadas"},
      "preictal":{"status":"DESCRIPTIVE ONLY","reason":"el generador no sintetiza una fase preictal"},
      "multicorpus_verdict":{"status":verdict,"reason":reason,"not_clinical_equivalence":True}}
    comparison_rows=[{"corpus":"CHB-MIT","protocol":"TSTR_SYN_TO_CHB","training_domain":"SYN","test_domain":"REAL_CHB",
      "status":chb_tstr.get("status","NOT EVALUABLE"),"macro_auc":chb_tstr.get("macro_auc"),"ci95_low":(chb_tstr.get("ci95") or [np.nan,np.nan])[0],"ci95_high":(chb_tstr.get("ci95") or [np.nan,np.nan])[1]}]
    for protocol_name,values in selected_protocols.items():
        comparison_rows.append({"corpus":"Siena","protocol":protocol_name,"training_domain":protocol_name.split("_TO_")[0],"test_domain":"REAL_SIENA",
          "status":tstr_status if protocol_name=="TSTR_SYN_TO_SIENA" else "REPORTED",**values})
    pd.DataFrame(comparison_rows).to_csv(OUT_DIR/"13_multicorpus_comparison.csv",index=False)
    (OUT_DIR/"08_siena_integrity_summary.json").write_text(json.dumps(SIENA["manifest"],indent=2,ensure_ascii=False),encoding="utf-8")
    (OUT_DIR/"13_multicorpus_final_summary.json").write_text(json.dumps(final,indent=2,ensure_ascii=False),encoding="utf-8")
    pd.DataFrame([
      {"claim":"transferencia SYN→CHB-MIT","status":chb_tstr.get("status","NOT EVALUABLE"),"evidence":chb_tstr.get("macro_auc")},
      {"claim":"transferencia SYN→Siena","status":tstr_status,"evidence":float(target.macro_mean)},
      {"claim":"morfología focal temporal frente a adultos","status":final["focal_temporal_external"]["status"],"evidence":within_fraction},
      {"claim":"ausencia generalizada frente a Siena","status":"NOT EVALUABLE","evidence":"sin cohorte compatible"},
      {"claim":"preictal sintetizado","status":"NOT SUPPORTED","evidence":"fuera del generador"},
      {"claim":"equivalencia clínica","status":"NOT SUPPORTED","evidence":"fuera del alcance"}]).to_csv(OUT_DIR/"13_multicorpus_claim_matrix.csv",index=False)
    print(json.dumps(final,indent=2,ensure_ascii=False))


In [ ]:
# BLOQUE 8 — endpoint Siena ictal frente a preictal, evaluable en los 14 sujetos
# Este endpoint es una prueba externa de detección. No implica síntesis ni predicción preictal.
SIENA_PREICTAL_ENDPOINT = 'ictal_vs_preictal_30_to_5_min'
SIENA_PREICTAL_MIN_SUBJECTS = 10
SIENA_PREICTAL_THRESHOLD = 0.5
SIENA_PREICTAL_BOOTSTRAPS = 2000

def external_scores(Xtr,ytr,Xte,yte,name,seed,threshold=SIENA_PREICTAL_THRESHOLD):
    Xtr=np.asarray(Xtr); ytr=np.asarray(ytr,dtype=int); Xte=np.asarray(Xte); yte=np.asarray(yte,dtype=int)
    if set(np.unique(ytr).tolist())!={0,1} or set(np.unique(yte).tolist())!={0,1}:
        return {'roc_auc':np.nan,'average_precision':np.nan,'sensitivity':np.nan,'specificity':np.nan}
    model=(make_pipeline(StandardScaler(),LogisticRegression(max_iter=2000,class_weight='balanced',random_state=seed))
           if name=='LR' else RandomForestClassifier(n_estimators=300,n_jobs=1,class_weight='balanced',random_state=seed))
    model.fit(Xtr,ytr); probability=model.predict_proba(Xte)[:,1]; predicted=probability>=threshold
    positive=yte==1; negative=yte==0
    return {'roc_auc':float(roc_auc_score(yte,probability)),
      'average_precision':float(average_precision_score(yte,probability)),
      'sensitivity':float(np.mean(predicted[positive])),'specificity':float(np.mean(~predicted[negative]))}

def run_siena_preictal_protocols(Fs_raw,Fs_z,ys,scenarios,Fpre_raw,Fpre_z,ypre,pidpre,Fchb_raw,Fchb_z,ychb):
    rows=[]; subjects=sorted(set(pidpre.tolist()))
    for subject in subjects:
        test=np.flatnonzero(pidpre==subject); train_all=np.flatnonzero(pidpre!=subject)
        if set(np.unique(ypre[test]).tolist())!={0,1}: raise RuntimeError(f'Siena monoclase en test: {subject}')
        if set(pidpre[test].tolist()) & set(pidpre[train_all].tolist()): raise RuntimeError(f'fuga LOSO: {subject}')
        seed=SEED+stable_int('siena_preictal_'+subject)%10000; rng=np.random.default_rng(seed)
        train=rng.choice(train_all,min(5000,len(train_all)),replace=False)
        if set(np.unique(ypre[train]).tolist())!={0,1}: raise RuntimeError(f'Siena monoclase en training: {subject}')
        syn_take=balanced(ys,2500,seed+1); chb_take=balanced(ychb,2500,seed+2)
        focal_pool=np.flatnonzero((ys==0)|((ys==1)&(scenarios=='focal_temporal')))
        focal_take=focal_pool[balanced(ys[focal_pool],2500,seed+3)]
        for mode,Fs,Fpre,Fchb in (('raw',Fs_raw,Fpre_raw,Fchb_raw),('zscore',Fs_z,Fpre_z,Fchb_z)):
            Xrtr,yrtr=Fpre[train],ypre[train]; Xrte,yrte=Fpre[test],ypre[test]
            Xsm,ysm=smote_training_only(Xrtr,yrtr,seed)
            specs={
              'SIENA_PREICTAL_R2R_LOSO':(Xrtr,yrtr,Xrte,yrte),
              'TSTR_SYN_TO_SIENA_PREICTAL':(Fs[syn_take],ys[syn_take],Xrte,yrte),
              'TSTR_FOCAL_SYN_TO_SIENA_PREICTAL':(Fs[focal_take],ys[focal_take],Xrte,yrte),
              'CHB_TO_SIENA_PREICTAL':(Fchb[chb_take],ychb[chb_take],Xrte,yrte),
              'CHB_PLUS_SYN_TO_SIENA_PREICTAL':(np.vstack([Fchb[chb_take],Fs[syn_take]]),np.concatenate([ychb[chb_take],ys[syn_take]]),Xrte,yrte),
              'SIENA_PLUS_SYN_PREICTAL_LOSO':(np.vstack([Xrtr,Fs[syn_take]]),np.concatenate([yrtr,ys[syn_take]]),Xrte,yrte),
              'SMOTE_TO_SIENA_PREICTAL':(Xsm,ysm,Xrte,yrte)}
            for model in ('LR','RF'):
                for protocol,(a,b,c,d) in specs.items():
                    scores=external_scores(a,b,c,d,model,seed)
                    rows.append({'subject':subject,'endpoint':SIENA_PREICTAL_ENDPOINT,'mode':mode,'model':model,
                      'protocol':protocol,**scores,'n_test':len(d),'n_test_ictal':int(np.sum(d==1)),'n_test_preictal':int(np.sum(d==0))})
    detail=pd.DataFrame(rows); summary=[]
    for keys,g in detail.groupby(['endpoint','mode','model','protocol']):
        for metric in ('roc_auc','average_precision','sensitivity','specificity'):
            valid=g[np.isfinite(g[metric])]; mean,lo,hi=bootstrap_subject(valid[metric],seed=SEED,n_boot=SIENA_PREICTAL_BOOTSTRAPS)
            summary.append({'endpoint':keys[0],'mode':keys[1],'model':keys[2],'protocol':keys[3],'metric':metric,
              'macro_mean':mean,'ci95_low':lo,'ci95_high':hi,'n_subjects':int(valid.subject.nunique()),
              'n_subjects_total':int(g.subject.nunique())})
    return detail,pd.DataFrame(summary)

def paired_subject_differences(detail):
    primary=detail[(detail['mode']=='zscore')&(detail.model=='RF')].pivot(index='subject',columns='protocol',values='roc_auc')
    comparisons=[
      ('SIENA_PLUS_SYN_minus_R2R','SIENA_PLUS_SYN_PREICTAL_LOSO','SIENA_PREICTAL_R2R_LOSO'),
      ('SIENA_PLUS_SYN_minus_SMOTE','SIENA_PLUS_SYN_PREICTAL_LOSO','SMOTE_TO_SIENA_PREICTAL'),
      ('CHB_PLUS_SYN_minus_CHB','CHB_PLUS_SYN_TO_SIENA_PREICTAL','CHB_TO_SIENA_PREICTAL'),
      ('TSTR_minus_R2R','TSTR_SYN_TO_SIENA_PREICTAL','SIENA_PREICTAL_R2R_LOSO')]
    rows=[]
    for label,a,b in comparisons:
        values=(primary[a]-primary[b]).dropna(); mean,lo,hi=bootstrap_subject(values,seed=SEED,n_boot=SIENA_PREICTAL_BOOTSTRAPS)
        status=('SUPPORTED' if len(values)>=SIENA_PREICTAL_MIN_SUBJECTS and mean>0.0 and lo>0.0 else
                ('NOT SUPPORTED' if len(values)>=SIENA_PREICTAL_MIN_SUBJECTS else 'NOT EVALUABLE'))
        rows.append({'comparison':label,'mean_auc_difference':mean,'ci95_low':lo,'ci95_high':hi,
          'subjects_improved':int((values>0).sum()),'n_subjects':int(len(values)),'status':status})
    return pd.DataFrame(rows)

if MODE=='siena_external':
    official_entries,_,official_edf=siena_official_index(verify_hashes=True)
    all_y=np.asarray(SIENA['y']); all_pid=np.asarray(SIENA['pid']).astype(str); all_meta=SIENA['meta'].copy()
    endpoint_mask=np.isin(all_y,[-1,1]); endpoint_index=np.flatnonzero(endpoint_mask)
    ypre=np.where(all_y[endpoint_mask]==1,1,0).astype(np.int8); pidpre=all_pid[endpoint_mask]
    mpre=all_meta.loc[endpoint_mask].reset_index(drop=True)
    expected_phase=np.where(ypre==1,'ictal','preictal')
    if not np.array_equal(mpre.phase.astype(str).to_numpy(),expected_phase): raise RuntimeError('inconsistencia fase/etiqueta Siena')
    duplicate_windows=int(mpre.duplicated(['subject_id','file','start_sample']).sum())
    subject_counts=mpre.groupby(['subject_id','phase']).size().unstack(fill_value=0)
    evaluable_subjects=subject_counts[(subject_counts.get('ictal',0)>0)&(subject_counts.get('preictal',0)>0)].index.tolist()
    strict_counts=all_meta[all_meta.phase.astype(str)=='interictal_strict'].groupby('subject_id').size()
    strict_subjects=sorted(strict_counts.index.astype(str).tolist())
    if len(evaluable_subjects)!=14 or duplicate_windows or len(strict_subjects)!=4:
        raise RuntimeError(f'contrato Siena 14 sujetos falló: evaluables={len(evaluable_subjects)}, duplicados={duplicate_windows}, estrictos={len(strict_subjects)}')
    Xpre=np.asarray(SIENA['X'])[endpoint_index]
    if not np.isfinite(Xpre).all(): raise RuntimeError('NaN/Inf en endpoint Siena ictal-preictal')
    Fpre_raw=Fsi_all[endpoint_index]
    Fpre_z=extract_features(Xpre,True,'SIENA_ictal_preictal_zscore')
    pre_detail,pre_summary=run_siena_preictal_protocols(Fs_raw,Fs_z,ys,scenarios,Fpre_raw,Fpre_z,ypre,pidpre,Fchb_raw,Fchb_z,ychb)
    paired=paired_subject_differences(pre_detail)
    target_pre=pre_summary[(pre_summary['mode']=='zscore')&(pre_summary.model=='RF')&
      (pre_summary.protocol=='TSTR_SYN_TO_SIENA_PREICTAL')&(pre_summary.metric=='roc_auc')].iloc[0]
    pre_status=('NOT EVALUABLE' if target_pre.n_subjects<SIENA_PREICTAL_MIN_SUBJECTS else
      ('SUPPORTED' if target_pre.macro_mean>0.5 and target_pre.ci95_low>0.5 else 'NOT SUPPORTED'))
    pre_detail.to_csv(OUT_DIR/'14_siena_ictal_preictal_loso.csv',index=False)
    pre_summary.to_csv(OUT_DIR/'14_siena_ictal_preictal_bootstrap.csv',index=False)
    paired.to_csv(OUT_DIR/'14_siena_ictal_preictal_paired_differences.csv',index=False)
    result_files=['14_siena_ictal_preictal_loso.csv','14_siena_ictal_preictal_bootstrap.csv','14_siena_ictal_preictal_paired_differences.csv']
    integrity={'schema_version':'1.0-siena-ictal-preictal','generated_utc':time.strftime('%Y-%m-%dT%H:%M:%SZ',time.gmtime()),
      'endpoint':SIENA_PREICTAL_ENDPOINT,'interpretation':'detección ictal frente a preictal; no síntesis ni predicción preictal',
      'criteria_frozen_in_notebook':{'minimum_subjects':SIENA_PREICTAL_MIN_SUBJECTS,'TSTR_supported':'macro AUROC > 0.5 and subject-bootstrap lower 95% > 0.5',
        'augmentation_supported':'paired subject-bootstrap lower 95% of AUROC difference > 0','classification_threshold':SIENA_PREICTAL_THRESHOLD},
      'source_integrity':{'official_sha256_entries_verified':len(official_entries),'official_edf':len(official_edf),
        'siena_artifact_hashes_verified':True,'synthetic_manifest_sha256':sha256_file(SYN_DIR/'generation_manifest.json')},
      'cohort':{'subjects_total':int(all_meta.subject_id.nunique()),'subjects_evaluable':len(evaluable_subjects),
        'evaluable_subject_ids':evaluable_subjects,'ictal_windows':int(np.sum(ypre==1)),'preictal_windows':int(np.sum(ypre==0)),
        'strict_interictal_subjects':strict_subjects,'strict_interictal_subject_count':len(strict_subjects),
        'duplicate_windows':duplicate_windows,'patient_leakage':False,'finite':True},
      'TSTR_RF_zscore':{'status':pre_status,'macro_auc':float(target_pre.macro_mean),
        'ci95':[float(target_pre.ci95_low),float(target_pre.ci95_high)],'n_subjects':int(target_pre.n_subjects)},
      'paired_differences':paired.to_dict('records'),
      'numeric_artifacts':{name:{'sha256':sha256_file(OUT_DIR/name),'bytes':int((OUT_DIR/name).stat().st_size)} for name in result_files}}
    (OUT_DIR/'14_siena_ictal_preictal_integrity.json').write_text(json.dumps(integrity,indent=2,ensure_ascii=False),encoding='utf-8')
    final_path=OUT_DIR/'13_multicorpus_final_summary.json'; updated=json.loads(final_path.read_text(encoding='utf-8'))
    updated['validation']['notebook_sha256']=sha256_file(PROJECT_ROOT/'1_1_EEGSynthesizer_VALIDATION.ipynb')
    updated['siena_TSTR_RF_zscore']['endpoint']='ictal_vs_interictal_strict_at_least_4h'
    updated['siena_ictal_vs_preictal_14']={'status':pre_status,'interpretation':'external seizure-detection stress test; not preictal synthesis',
      'macro_auc':float(target_pre.macro_mean),'ci95':[float(target_pre.ci95_low),float(target_pre.ci95_high)],
      'n_subjects':int(target_pre.n_subjects),'paired_differences':paired.to_dict('records')}
    if pre_status=='SUPPORTED':
        updated['multicorpus_verdict']={'status':'VIABLE REFORZADO CON LIMITACIONES',
          'reason':'SYN→Siena está respaldado para detección ictal frente a preictal en 14 sujetos; el endpoint interictal estricto sigue limitado a cuatro sujetos y no se afirma equivalencia clínica.',
          'not_clinical_equivalence':True}
    else:
        updated['multicorpus_verdict']={'status':'VIABLE CON LIMITACIONES',
          'reason':'La evaluación ictal frente a preictal incluye 14 sujetos, pero SYN→Siena no supera el criterio prospectivo; el endpoint interictal estricto sigue limitado a cuatro sujetos.',
          'not_clinical_equivalence':True}
    final_path.write_text(json.dumps(updated,indent=2,ensure_ascii=False),encoding='utf-8')
    claims=pd.read_csv(OUT_DIR/'13_multicorpus_claim_matrix.csv')
    claims=claims[claims['claim'].astype(str)!='transferencia SYN→Siena']
    claims=pd.concat([claims,pd.DataFrame([
      {'claim':'transferencia SYN→Siena: ictal vs interictal estricto','status':tstr_status,'evidence':float(target.macro_mean)},
      {'claim':'transferencia SYN→Siena: ictal vs preictal (14 sujetos)','status':pre_status,'evidence':float(target_pre.macro_mean)}])],ignore_index=True)
    claims.to_csv(OUT_DIR/'13_multicorpus_claim_matrix.csv',index=False)
    print(json.dumps({'endpoint':SIENA_PREICTAL_ENDPOINT,'status':pre_status,'macro_auc':float(target_pre.macro_mean),
      'ci95':[float(target_pre.ci95_low),float(target_pre.ci95_high)],'n_subjects':int(target_pre.n_subjects),
      'strict_interictal_subjects':strict_subjects,'paired_differences':paired.to_dict('records')},indent=2,ensure_ascii=False))
    del Xpre,Fpre_raw,Fpre_z; gc.collect()


if MODE=="siena_external":
    record_stage("S5_siena_raw","SUPPORTED",{"official_checksums":SIENA_CACHE/"SHA256SUMS.txt"},
                 {"official_entries":58,"official_edf":41,"raw_download":SIENA.get("raw_status",{})})
    record_stage("S6_siena_derived","SUPPORTED",{"siena_manifest":REAL_DIR/"siena_manifest.json",
                 "annotation_audit":REAL_DIR/"siena_annotation_audit.csv"},
                 {"subjects":int(SIENA["meta"].subject_id.nunique()),"windows":int(len(SIENA["y"]))})
    record_stage("S7_external_results","SUPPORTED",{"multicorpus_summary":OUT_DIR/"13_multicorpus_final_summary.json",
                 "ictal_preictal_integrity":OUT_DIR/"14_siena_ictal_preictal_integrity.json"},
                 {"interpretation":"operational research validation; not clinical equivalence"})
    print("Reanudación registrada en",STATE_PATH)


In [ ]:
# BLOQUE 9 — productos reproducibles para tesis y artículo
from IPython.display import Image, Markdown, display
import matplotlib.pyplot as plt

PUBLICATION_SOURCE_FILES = [
    "02_fidelity_featurewise.csv",
    "02b_morphology_coverage.csv",
    "03b_cross_domain_bootstrap.csv",
    "05_final_summary.json",
    "07_generalized_symmetry_summary.json",
    "09_siena_fidelity_featurewise.csv",
    "09_siena_morphology_coverage.csv",
    "12_siena_laterality_audit.csv",
    "13_multicorpus_final_summary.json",
    "14_siena_ictal_preictal_integrity.json",
    "14_siena_ictal_preictal_paired_differences.csv",
]

PUBLICATION_FIGURES = [
    "16_fig_auroc_forest",
    "16_fig_syn_effect",
    "16_fig_morphology_fidelity",
    "16_fig_signal_comparison",
    "16_fig_psd_multicorpus",
    "16_fig_symmetry_degradation",
]

PUBLICATION_COLORS = {"SYN":"#0072B2", "CHB-MIT":"#D55E00", "Siena":"#009E73",
                      "SUPPORTED":"#009E73", "NOT SUPPORTED":"#D55E00",
                      "NOT EVALUABLE":"#7F7F7F", "REPORTED":"#0072B2"}
PUBLICATION_STATUS_LABELS = {"SUPPORTED":"RESPALDADO","NOT SUPPORTED":"CRITERIO COMPLETO NO ALCANZADO",
                             "NOT EVALUABLE":"EXPLORATORIO (n < 10)","REPORTED":"REFERENCIA REAL-REAL"}

def save_publication_figure(fig, stem):
    png=OUT_DIR/f"{stem}.png"; pdf=OUT_DIR/f"{stem}.pdf"
    fig.savefig(png,dpi=220,bbox_inches="tight",facecolor="white")
    fig.savefig(pdf,bbox_inches="tight",facecolor="white",
                metadata={"Creator":"EEGSynthesizer","CreationDate":None,"ModDate":None})
    plt.close(fig); display(Image(filename=str(png)))
    return [png,pdf]

def publication_medoid(batch):
    features=np.asarray([one_feature(window,zscore=True) for window in batch],dtype=np.float64)
    center=np.median(features,axis=0)
    scale=stats.median_abs_deviation(features,axis=0,scale="normal")+1e-9
    return int(np.argmin(np.sum(((features-center)/scale)**2,axis=1)))

def publication_ictal_batches(n_pool=200):
    rng=np.random.default_rng(20260804)
    Xsyn=np.load(SYN_DIR/"X_test.npy",mmap_mode="r"); ysyn=np.load(SYN_DIR/"y_test.npy"); msyn=pd.read_csv(SYN_DIR/"window_metadata_test.csv")
    Xchb=np.load(REAL_DIR/"X_real.npy",mmap_mode="r"); ychb=np.load(REAL_DIR/"y_real.npy"); pchb=np.load(REAL_DIR/"pid_real.npy"); mchb=pd.read_csv(REAL_DIR/"real_window_metadata.csv")
    Xsiena=np.load(REAL_DIR/"siena_X.npy",mmap_mode="r"); ysiena=np.load(REAL_DIR/"siena_y.npy"); msiena=pd.read_csv(REAL_DIR/"siena_window_metadata.csv")
    external_chb=~np.isin(pchb.astype(str),sorted(DEV_CASES))
    specifications=[("SYN",Xsyn,np.flatnonzero((ysyn==1)&(msyn.ictal_fraction.to_numpy()>=0.999)),True),
                    ("CHB-MIT",Xchb,np.flatnonzero((ychb==1)&external_chb&(mchb.ictal_fraction.to_numpy()>=0.999)),False),
                    ("Siena",Xsiena,np.flatnonzero((ysiena==1)&(msiena.ictal_fraction.to_numpy()>=0.999)),False)]
    batches={}; selected_indices={}
    for domain,array,indices,is_referential in specifications:
        if len(indices)==0: raise RuntimeError(f"No hay ventanas ictales para {domain}")
        selected=rng.choice(indices,min(n_pool,len(indices)),replace=False)
        batch=np.asarray(array[selected],dtype=np.float32)
        if is_referential: batch=synthetic_to_bipolar(batch)
        if batch.shape[1:]!=(WIN_PTS,len(BIPOLAR)): raise RuntimeError(f"Forma inesperada para {domain}: {batch.shape}")
        if not np.isfinite(batch).all(): raise RuntimeError(f"NaN/Inf en lote gráfico {domain}")
        batches[domain]=batch; selected_indices[domain]=selected.astype(int)
    return batches,selected_indices

missing_publication_sources=[name for name in PUBLICATION_SOURCE_FILES if not (OUT_DIR/name).exists()]
heavy_publication_sources=[SYN_DIR/"X_test.npy",SYN_DIR/"y_test.npy",SYN_DIR/"window_metadata_test.csv",
                           REAL_DIR/"X_real.npy",REAL_DIR/"y_real.npy",REAL_DIR/"pid_real.npy",REAL_DIR/"real_window_metadata.csv",
                           REAL_DIR/"siena_X.npy",REAL_DIR/"siena_y.npy",REAL_DIR/"siena_window_metadata.csv"]
missing_heavy=[path.name for path in heavy_publication_sources if not path.exists()]

if missing_publication_sources:
    print("Productos de publicación: NOT EVALUABLE — faltan resultados:",missing_publication_sources)
else:
    summary_chb=json.loads((OUT_DIR/"05_final_summary.json").read_text(encoding="utf-8"))
    summary_multi=json.loads((OUT_DIR/"13_multicorpus_final_summary.json").read_text(encoding="utf-8"))
    summary_pre=json.loads((OUT_DIR/"14_siena_ictal_preictal_integrity.json").read_text(encoding="utf-8"))
    symmetry=json.loads((OUT_DIR/"07_generalized_symmetry_summary.json").read_text(encoding="utf-8"))
    paired=pd.read_csv(OUT_DIR/"14_siena_ictal_preictal_paired_differences.csv")
    laterality=pd.read_csv(OUT_DIR/"12_siena_laterality_audit.csv"); n_laterality=int(laterality.evaluable.astype(bool).sum())
    n_correct_laterality=int(laterality.loc[laterality.evaluable.astype(bool),"correct"].astype(bool).sum())

    chb=summary_chb["claims"]["external_TSTR_RF_zscore"]
    pre=summary_pre["TSTR_RF_zscore"]
    strict=summary_multi["siena_TSTR_RF_zscore"]
    focal=summary_multi["focal_temporal_external"]
    n_covered_features=int(round(focal["features_within_real_real_q95_fraction"]*len(FEATURE_NAMES)))
    improvement=paired.loc[paired.comparison=="CHB_PLUS_SYN_minus_CHB"].iloc[0]
    publication_table=pd.DataFrame([
      {"Corpus":"CHB-MIT","Endpoint":"ictal vs. interictal","Estimando":"AUROC TSTR","Estimación":chb["macro_auc"],
       "IC95_inf":chb["ci95"][0],"IC95_sup":chb["ci95"][1],"Unidad/N":f"{chb['n_subjects']} sujetos","Resultado":"Respaldado: IC95 % AUROC > 0.5"},
      {"Corpus":"Siena","Endpoint":"ictal vs. preictal (30-5 min)","Estimando":"AUROC TSTR","Estimación":pre["macro_auc"],
       "IC95_inf":pre["ci95"][0],"IC95_sup":pre["ci95"][1],"Unidad/N":f"{pre['n_subjects']} sujetos","Resultado":"Respaldado: IC95 % AUROC > 0.5"},
      {"Corpus":"Siena","Endpoint":"ictal vs. interictal estricto","Estimando":"AUROC TSTR","Estimación":strict["macro_auc"],
       "IC95_inf":strict["ci95"][0],"IC95_sup":strict["ci95"][1],"Unidad/N":f"{strict['n_subjects']} sujetos","Resultado":"Estimación exploratoria: n < 10"},
      {"Corpus":"Siena","Endpoint":"ictal vs. preictal (30-5 min)","Estimando":"Delta AUROC CHB+SYN-CHB","Estimación":improvement.mean_auc_difference,
       "IC95_inf":improvement.ci95_low,"IC95_sup":improvement.ci95_high,"Unidad/N":f"{int(improvement.n_subjects)} sujetos","Resultado":"Beneficio respaldado: IC95 % diferencia > 0"},
      {"Corpus":"Siena","Endpoint":"morfología focal temporal","Estimando":"fracción dentro de REAL-REAL q95","Estimación":focal["features_within_real_real_q95_fraction"],
       "IC95_inf":np.nan,"IC95_sup":np.nan,"Unidad/N":f"{len(FEATURE_NAMES)} rasgos","Resultado":f"Concordancia {n_covered_features}/{len(FEATURE_NAMES)}; criterio completo {len(FEATURE_NAMES)}/{len(FEATURE_NAMES)}"},
      {"Corpus":"Siena","Endpoint":"lateralidad focal temporal","Estimando":"exactitud de lateralidad","Estimación":focal["laterality_accuracy"],
       "IC95_inf":np.nan,"IC95_sup":np.nan,"Unidad/N":f"{n_laterality} sujetos evaluables","Resultado":f"Descriptivo: concordancia {n_correct_laterality}/{n_laterality}; sin umbral confirmatorio"},
    ])
    publication_table.to_csv(OUT_DIR/"16_publication_multicorpus_table.csv",index=False)
    (OUT_DIR/"16_publication_multicorpus_table.tex").write_text(
        publication_table.to_latex(index=False,float_format=lambda value:f"{value:.4f}",na_rep="--"),encoding="utf-8")
    display(Markdown("## Tabla comparativa multicorpus"))
    display(publication_table.style.format({"Estimación":"{:.4f}","IC95_inf":"{:.4f}","IC95_sup":"{:.4f}"},na_rep="--"))

    # Forest plot de AUROC. Los endpoints se mantienen explícitos porque no son intercambiables.
    rr=summary_multi["siena_protocols_RF_zscore"]["CHB_TO_SIENA_REAL_REAL"]
    forest=pd.DataFrame([
      {"label":"CHB-MIT\nTSTR SYN→REAL","mean":chb["macro_auc"],"low":chb["ci95"][0],"high":chb["ci95"][1],"n":chb["n_subjects"],"status":chb["status"]},
      {"label":"Siena preictal\nTSTR SYN→REAL","mean":pre["macro_auc"],"low":pre["ci95"][0],"high":pre["ci95"][1],"n":pre["n_subjects"],"status":pre["status"]},
      {"label":"Siena interictal estricto\nTSTR SYN→REAL","mean":strict["macro_auc"],"low":strict["ci95"][0],"high":strict["ci95"][1],"n":strict["n_subjects"],"status":strict["status"]},
      {"label":"Siena interictal estricto\nCHB→Siena REAL–REAL","mean":rr["macro_auc"],"low":rr["ci95_low"],"high":rr["ci95_high"],"n":rr["n_subjects"],"status":"REPORTED"},
    ])
    fig,ax=plt.subplots(figsize=(9,5.2)); y=np.arange(len(forest))[::-1]
    for yi,row in zip(y,forest.itertuples()):
        color=PUBLICATION_COLORS.get(row.status,"#0072B2")
        ax.errorbar(row.mean,yi,xerr=[[row.mean-row.low],[row.high-row.mean]],fmt="o",ms=8,capsize=4,color=color,lw=2)
        ax.text(min(0.82,row.high+0.015),yi,f"n={row.n}; {PUBLICATION_STATUS_LABELS.get(row.status,row.status)}",va="center",fontsize=9,color=color)
    ax.axvline(.5,color="black",ls="--",lw=1,label="azar: AUROC = 0.5")
    ax.set(yticks=y,yticklabels=forest.label,xlim=(.35,1.02),xlabel="AUROC macro e IC95 % por sujeto",
           title="Transferencia externa multicorpus")
    ax.grid(axis="x",alpha=.25); ax.legend(loc="upper center",bbox_to_anchor=(.5,-.18)); fig.tight_layout()
    save_publication_figure(fig,"16_fig_auroc_forest")

    # Efectos pareados de añadir SYN en el endpoint Siena ictal-preictal.
    effect_order=["CHB_PLUS_SYN_minus_CHB","SIENA_PLUS_SYN_minus_R2R","SIENA_PLUS_SYN_minus_SMOTE","TSTR_minus_R2R"]
    effects=paired.set_index("comparison").loc[effect_order].reset_index()
    effect_labels={"CHB_PLUS_SYN_minus_CHB":"CHB+SYN - CHB","SIENA_PLUS_SYN_minus_R2R":"Siena+SYN - R2R",
                   "SIENA_PLUS_SYN_minus_SMOTE":"Siena+SYN - SMOTE","TSTR_minus_R2R":"TSTR - R2R"}
    fig,ax=plt.subplots(figsize=(9,4.8)); y=np.arange(len(effects))[::-1]
    for yi,row in zip(y,effects.itertuples()):
        color=PUBLICATION_COLORS.get(row.status,"#7F7F7F")
        ax.errorbar(row.mean_auc_difference,yi,
                    xerr=[[row.mean_auc_difference-row.ci95_low],[row.ci95_high-row.mean_auc_difference]],
                    fmt="o",ms=8,capsize=4,color=color,lw=2)
        interpretation=("IC95 % > 0" if row.status=="SUPPORTED" else "IC95 % incluye 0")
        ax.text(row.ci95_high+.004,yi,f"{row.mean_auc_difference:+.3f}; {row.subjects_improved}/{row.n_subjects}; {interpretation}",va="center",fontsize=9)
    ax.axvline(0,color="black",ls="--",lw=1); ax.set_xlim(float(effects.ci95_low.min())-.01,float(effects.ci95_high.max())+.075); ax.set(yticks=y,yticklabels=[effect_labels[x] for x in effects.comparison],
      xlabel="Diferencia pareada de AUROC e IC95 %",title="Efecto incremental de SYN - Siena ictal frente a preictal")
    ax.grid(axis="x",alpha=.25); fig.tight_layout(); save_publication_figure(fig,"16_fig_syn_effect")

    # Cobertura morfológica y fidelidad por corpus, clase y característica.
    chb_cov=pd.read_csv(OUT_DIR/"02b_morphology_coverage.csv"); siena_cov=pd.read_csv(OUT_DIR/"09_siena_morphology_coverage.csv")
    chb_fid=pd.read_csv(OUT_DIR/"02_fidelity_featurewise.csv"); siena_fid=pd.read_csv(OUT_DIR/"09_siena_fidelity_featurewise.csv")
    fig,axes=plt.subplots(3,2,figsize=(17,11.5),constrained_layout=True)
    panels=[(chb_cov,"real_inside_syn_05_95","CHB-MIT: REAL dentro de SYN [P5,P95]","coverage"),
            (siena_cov,"real_inside_syn_05_95","Siena: REAL dentro de SYN [P5,P95]","coverage"),
            (chb_cov,"syn_inside_real_05_95","CHB-MIT: SYN dentro de REAL [P5,P95]","coverage"),
            (siena_cov,"syn_inside_real_05_95","Siena: SYN dentro de REAL [P5,P95]","coverage"),
            (chb_fid,"cliffs_delta","CHB-MIT: |Cliff's delta|","effect"),
            (siena_fid,"cliffs_delta","Siena: |Cliff's delta|","effect")]
    for ax,(frame,value,title,kind) in zip(axes.flat,panels):
        matrix=frame.pivot(index="group",columns="feature",values=value).reindex(columns=FEATURE_NAMES)
        values=np.abs(matrix.to_numpy(dtype=float)) if kind=="effect" else matrix.to_numpy(dtype=float)
        image=ax.imshow(values,aspect="auto",vmin=0,vmax=1,cmap="viridis_r" if kind=="effect" else "viridis")
        ax.set(yticks=np.arange(len(matrix.index)),yticklabels=matrix.index,xticks=np.arange(len(matrix.columns)),xticklabels=matrix.columns,ylabel="Escenario SYN")
        ax.tick_params(axis="x",rotation=75,labelsize=8); ax.set_title(title)
        colorbar=fig.colorbar(image,ax=ax,fraction=.025,pad=.02); colorbar.set_label("mayor = mejor" if kind=="coverage" else "menor = mejor")
    fig.suptitle("Cobertura bidireccional y magnitud de discrepancia SYN-REAL",fontsize=14)
    save_publication_figure(fig,"16_fig_morphology_fidelity")

    # Simetría limpia y observable después de degradación.
    symmetry_labels=["Antes de degradación","Después de degradación"]; symmetry_total=int(symmetry["n_generalized_events"])
    symmetry_rates=[symmetry["clean_nondegraded"]["pass_rate"],symmetry["final_all_channels"]["pass_rate"]]
    symmetry_counts=[symmetry["clean_nondegraded"]["pass_count"],symmetry["final_all_channels"]["pass_count"]]
    symmetry_status=[symmetry["clean_nondegraded"]["status"],symmetry["final_all_channels"]["status"]]
    symmetry_display=["CRITERIO INTERNO\nCUMPLIDO","CRITERIO 90 %\nNO ALCANZADO"]
    fig,ax=plt.subplots(figsize=(7.5,4.8)); bars=ax.bar(symmetry_labels,symmetry_rates,
      color=[PUBLICATION_COLORS.get(status,"#7F7F7F") for status in symmetry_status],width=.6)
    ax.axhline(symmetry["acceptance_rate"],color="black",ls="--",lw=1,label=f"criterio: {symmetry['acceptance_rate']:.0%}")
    for bar,rate,count,status,display_status in zip(bars,symmetry_rates,symmetry_counts,symmetry_status,symmetry_display):
        ypos=(symmetry["acceptance_rate"]-.04 if rate>=symmetry["acceptance_rate"] else rate-.035); valign="top"; color="white"
        ax.text(bar.get_x()+bar.get_width()/2,ypos,f"{count}/{symmetry_total} = {rate:.1%}\n{display_status}",
                ha="center",va=valign,color=color,fontweight="bold")
    ax.text(.5,.42,f"Degradación frontal: {symmetry['events_with_frontal_pair_degradation']} eventos\nFallos finales: {symmetry_total-symmetry_counts[1]}",
            transform=ax.transAxes,ha="center",va="center",fontsize=9,bbox={"facecolor":"white","alpha":.92,"edgecolor":"#B0B0B0"})
    ax.set_ylim(0,1.13); ax.set_ylabel("Proporción que cumple el criterio"); ax.set_title("Robustez de la simetría ante degradación de adquisición")
    ax.legend(loc="lower left"); ax.grid(axis="y",alpha=.2); fig.tight_layout(); save_publication_figure(fig,"16_fig_symmetry_degradation")

    generated_signal_outputs=[]
    if missing_heavy:
        print("Se omiten señales/PSD multicorpus; faltan artefactos pesados:",missing_heavy)
    else:
        batches,publication_indices=publication_ictal_batches()
        representative_positions={domain:publication_medoid(batch) for domain,batch in batches.items()}
        representatives={domain:batch[representative_positions[domain]] for domain,batch in batches.items()}
        colors={domain:PUBLICATION_COLORS[domain] for domain in representatives}
        t=np.arange(WIN_PTS)/FS; traces={}; channels={}; representative_rows=[]
        metadata_frames={"SYN":pd.read_csv(SYN_DIR/"window_metadata_test.csv"),"CHB-MIT":pd.read_csv(REAL_DIR/"real_window_metadata.csv"),
                         "Siena":pd.read_csv(REAL_DIR/"siena_window_metadata.csv")}
        for domain,window in representatives.items():
            channel=int(np.argmax(np.ptp(window,axis=0))); channels[domain]=channel
            traces[domain]=window[:,channel]-np.median(window[:,channel])
        raw_ptp={domain:float(np.ptp(trace)) for domain,trace in traces.items()}
        normalized={domain:trace/(np.percentile(np.abs(trace),95)+1e-12) for domain,trace in traces.items()}
        for domain,batch in batches.items():
            position=representative_positions[domain]; source_index=int(publication_indices[domain][position]); meta=metadata_frames[domain].iloc[source_index]
            subject=str(meta.get("subject_id",meta.get("patient_id",""))); subject=(f"p{int(subject):04d}" if domain=="SYN" else subject); event=str(meta.get("event_id","")); start=float(meta.get("start_s",meta.get("window_start_s",np.nan)))
            pool_ptp=np.max(np.ptp(batch,axis=1),axis=1); median_channel_ptp=float(np.median(np.ptp(representatives[domain],axis=0)))
            representative_rows.append({"domain":domain,"source_array_index":source_index,"subject_or_patient":subject,"event_id":event,
              "start_s":start,"ictal_fraction":float(meta.get("ictal_fraction",np.nan)),"scenario":str(meta.get("scenario","real_ictal")),
              "displayed_derivation":BIPOLAR[channels[domain]][0],"displayed_raw_ptp_uv":raw_ptp[domain],"median_channel_ptp_uv":median_channel_ptp,
              "max_ptp_percentile_within_pool":float(np.mean(pool_ptp<=raw_ptp[domain])),"pool_size":len(batch),"selection_seed":20260804,
              "selection_rule":"medoid in robust z-scored morphology features; maximum-PTP derivation displayed",
              "display_transform":"median centered; divided by channel 95th percentile absolute amplitude"})
        representative_provenance=pd.DataFrame(representative_rows); representative_provenance.to_csv(OUT_DIR/"16_publication_representative_signals.csv",index=False)
        limit=max(np.percentile(np.abs(trace),99.5) for trace in normalized.values())*1.08
        fig,axes=plt.subplots(3,1,figsize=(10,7),sharex=True,sharey=True)
        for ax,domain in zip(axes,["SYN","CHB-MIT","Siena"]):
            ax.plot(t,normalized[domain],color=colors[domain],lw=1)
            row=representative_provenance[representative_provenance.domain==domain].iloc[0]
            ax.set_title(f"{domain} ({row.subject_or_patient}, t={row.start_s:.1f} s) - medoide ictal completo; {BIPOLAR[channels[domain]][0]}; PTP máx.={raw_ptp[domain]:.1f} µV")
            ax.set_ylabel("Amplitud\nnormalizada"); ax.grid(alpha=.2); ax.set_ylim(-limit,limit)
        axes[-1].set_xlabel("Tiempo (s)"); fig.suptitle("Morfología ictal: ventanas 100 % ictales, normalizadas solo para visualización",fontsize=14)
        fig.tight_layout(); generated_signal_outputs+=save_publication_figure(fig,"16_fig_signal_comparison")

        fig,ax=plt.subplots(figsize=(9,5.2))
        for domain,batch in batches.items():
            frequency,power=signal.welch(batch,fs=FS,nperseg=256,axis=1)
            spectrum=np.median(power,axis=(0,2)); mask=(frequency>=1)&(frequency<=30)
            ax.semilogy(frequency[mask],spectrum[mask],color=colors[domain],lw=2,label=domain)
        ax.set(xlabel="Frecuencia (Hz)",ylabel="PSD mediana (µV²/Hz)",title="PSD absoluta ictal sin normalización - 200 ventanas completas, montaje común, 250 Hz")
        ax.grid(alpha=.25); ax.legend(); fig.tight_layout(); generated_signal_outputs+=save_publication_figure(fig,"16_fig_psd_multicorpus")

    captions=pd.DataFrame([
      {"archivo":"16_fig_auroc_forest","titulo":"Transferencia externa multicorpus","interpretacion":"AUROC macro e IC95 % bootstrap por sujeto; cada endpoint se interpreta por separado.","alcance_inferencial":"Utilidad predictiva externa por corpus; CHB-MIT pediátrico y Siena adulto se mantienen como dominios independientes."},
      {"archivo":"16_fig_syn_effect","titulo":"Efecto incremental de SYN","interpretacion":"Diferencias pareadas de AUROC e IC95 % bootstrap por sujeto en Siena ictal frente a preictal.","alcance_inferencial":"El beneficio confirmado corresponde a CHB+SYN frente a CHB; en las demás comparaciones el IC95 % incluye cero."},
      {"archivo":"16_fig_morphology_fidelity","titulo":"Cobertura bidireccional y discrepancia morfológica","interpretacion":"REAL dentro de SYN, SYN dentro de REAL y magnitud absoluta de Cliff por rasgo y escenario.","alcance_inferencial":"Concordancia morfológica por rasgos entre escenarios paramétricos SYN y el dominio ictal real."},
      {"archivo":"16_fig_signal_comparison","titulo":"Señales ictales representativas","interpretacion":"Medoides morfoespectrales de ventanas con ocupación ictal completa; amplitud normalizada solo para visualización.","alcance_inferencial":"Comparación descriptiva trazable, complementaria a las métricas agregadas; índices y amplitudes crudas constan en 16_publication_representative_signals.csv."},
      {"archivo":"16_fig_psd_multicorpus","titulo":"PSD ictal multicorpus","interpretacion":"PSD absoluta mediana de 200 ventanas completamente ictales por corpus, 18 derivaciones, montaje común y 250 Hz.","alcance_inferencial":"Comparación espectral absoluta que conserva las diferencias de escala entre dominios y sistemas de adquisición."},
      {"archivo":"16_fig_symmetry_degradation","titulo":"Robustez de simetría ante degradación","interpretacion":"Separa la regla morfológica limpia de la señal finalmente observable y cuantifica la degradación frontal.","alcance_inferencial":"Control interno de robustez morfológica bajo degradación de adquisición programada."},
    ])
    captions.to_csv(OUT_DIR/"16_publication_captions.csv",index=False)
    figure_audit=pd.DataFrame([
      {"figura":"16_fig_auroc_forest","fuentes":"05_final_summary.json; 13_multicorpus_final_summary.json; 14_siena_ictal_preictal_integrity.json","verificacion":"estimaciones, IC95 y n coinciden con fuentes","uso_permitido":"transferencia externa por endpoint","alcance_inferencial":"utilidad predictiva por corpus y endpoint, sin combinar cohortes","estado":"SUPPORTED"},
      {"figura":"16_fig_syn_effect","fuentes":"14_siena_ictal_preictal_paired_differences.csv","verificacion":"diferencias pareadas, IC95 y sujetos mejorados coinciden","uso_permitido":"efecto del aumento con datos sintéticos en Siena ictal-preictal","alcance_inferencial":"beneficio confirmado para CHB+SYN frente a CHB; demás comparaciones inconclusas","estado":"SUPPORTED"},
      {"figura":"16_fig_morphology_fidelity","fuentes":"02/09 fidelity_featurewise y morphology_coverage","verificacion":"19 rasgos; cobertura bidireccional; valor absoluto de Cliff","uso_permitido":"describir cobertura y desplazamiento morfológico","alcance_inferencial":"concordancia por rasgos entre escenarios SYN y dominio ictal real","estado":"SUPPORTED"},
      {"figura":"16_fig_signal_comparison","fuentes":"arrays y metadatos SYN/CHB-MIT/Siena con SHA256","verificacion":"ventanas 100 % ictales; medoide determinista; procedencia guardada","uso_permitido":"comparación descriptiva de morfología","alcance_inferencial":"ejemplo trazable complementario a métricas morfológicas agregadas","estado":"SUPPORTED"},
      {"figura":"16_fig_psd_multicorpus","fuentes":"mismos lotes completamente ictales que la figura de señales","verificacion":"mediana de 200 ventanas x 18 derivaciones; PSD absoluta","uso_permitido":"comparar forma espectral y escala","alcance_inferencial":"comparación absoluta entre dominios y sistemas de adquisición","estado":"SUPPORTED"},
      {"figura":"16_fig_symmetry_degradation","fuentes":"07_generalized_symmetry_summary.json","verificacion":"denominador dinámico; conteos y umbral coinciden","uso_permitido":"control morfológico interno antes/después de degradación","alcance_inferencial":"robustez del escenario generalizado frente a degradación programada","estado":"SUPPORTED"},
    ]); figure_audit.to_csv(OUT_DIR/"16_publication_figure_audit.csv",index=False)

    output_paths=[OUT_DIR/"16_publication_multicorpus_table.csv",OUT_DIR/"16_publication_multicorpus_table.tex",OUT_DIR/"16_publication_captions.csv",
                  OUT_DIR/"16_publication_figure_audit.csv"]
    if (OUT_DIR/"16_publication_representative_signals.csv").exists(): output_paths.append(OUT_DIR/"16_publication_representative_signals.csv")
    for stem in PUBLICATION_FIGURES:
        for suffix in (".png",".pdf"):
            candidate=OUT_DIR/f"{stem}{suffix}"
            if candidate.exists(): output_paths.append(candidate)
    source_paths=[OUT_DIR/name for name in PUBLICATION_SOURCE_FILES]+heavy_publication_sources
    publication_manifest={
      "schema_version":"1.1-publication-products",
      "generated_utc":time.strftime("%Y-%m-%dT%H:%M:%SZ",time.gmtime()),
      "validator_notebook_sha256":sha256_file(PROJECT_ROOT/"1_1_EEGSynthesizer_VALIDATION.ipynb"),
      "sources":{str(path.relative_to(PROJECT_ROOT)).replace("\\","/"):{"bytes":path.stat().st_size,"sha256":sha256_file(path)} for path in source_paths if path.exists()},
      "outputs":{path.name:{"bytes":path.stat().st_size,"sha256":sha256_file(path)} for path in output_paths},
      "selection":{"eligibility":"ictal_fraction >= 0.999 in every corpus; CHB-MIT development subjects excluded",
                   "waveforms":"medoid in robust z-scored feature space; maximum-PTP bipolar derivation displayed after medoid selection; exact provenance in 16_publication_representative_signals.csv",
                   "psd":"absolute median over 200 deterministic fully ictal windows and 18 bipolar derivations; no amplitude normalization"},
      "scope":"productos descriptivos y confirmatorios para publicación; los CSV/JSON numéricos son la fuente inferencial",
      "inference_scope":"plausibilidad morfológica operacional, cobertura por rasgos y utilidad de transferencia externa",
      "formal_clinical_equivalence_design_performed":False,
    }
    atomic_write_json(OUT_DIR/"16_publication_manifest.json",publication_manifest)
    display(Markdown("**Productos de publicación: SUPPORTED.** Se guardaron tabla, leyendas, PNG, PDF y manifiesto con hashes."))


## Productos gráficos reproducibles

El bloque anterior reconstruye estas figuras únicamente desde artefactos verificados.
Las versiones PDF, la tabla LaTeX, las leyendas, la auditoría por figura, la
procedencia exacta de las señales representativas y el manifiesto SHA-256 están en
`dataset_doctorado_final/validation_q1_assets/`.

### Transferencia externa

![Forest plot AUROC](dataset_doctorado_final/validation_q1_assets/16_fig_auroc_forest.png)

### Efecto de incorporar SYN

![Efectos pareados](dataset_doctorado_final/validation_q1_assets/16_fig_syn_effect.png)

### Cobertura y fidelidad morfológica

![Cobertura y fidelidad](dataset_doctorado_final/validation_q1_assets/16_fig_morphology_fidelity.png)

### Señales y espectros multicorpus

![Señales representativas](dataset_doctorado_final/validation_q1_assets/16_fig_signal_comparison.png)

![PSD multicorpus](dataset_doctorado_final/validation_q1_assets/16_fig_psd_multicorpus.png)

### Simetría y degradación

![Simetría](dataset_doctorado_final/validation_q1_assets/16_fig_symmetry_degradation.png)

Estas figuras documentan plausibilidad morfológica operacional, concordancia por
rasgos, diferencias entre dominios y utilidad de transferencia. La equivalencia
clínica corresponde a un diseño diferente con márgenes clínicos predefinidos.
